# WSIMOD staged calibration framework (Phase A–B)

**Framework scope:** This executable notebook implements the two-stage calibration workflow described in the dissertation: evidence-informed regionalisation-factor construction (Section 2.4), separate calibration-domain experiments (Phase A; Section 2.5), combined intensity-factor testing and selection (Phase B; Section 2.6), then multi-site/multi-metric assessment and untouched validation (Section 2.7).

**Deliberate boundary:** The selected configuration is the best-tested Phase-B configuration in the predefined grid. Catchment-level results diagnose spatial consistency; they do not initiate a further parameter-adjustment stage.

**Reproducibility:** Each executable cell begins with a concise comment stating what it does and which dissertation section it implements. Source hashes, parameter change logs, generated configuration files and output completeness checks remain active.


In [ ]:
from pathlib import Path
import os
import sys

# ------------------------------------------------------------------
# PORTABLE USER SETTINGS — change only WSIMOD_STAGED_CALIBRATION_FRAMEWORK_ROOT if required.
# ------------------------------------------------------------------
# The same value can be supplied outside Jupyter, for example:
#   set WSIMOD_STAGED_CALIBRATION_FRAMEWORK_ROOT=D:\my_wsimod_reproduction
# The project root must contain wsimod-staged-calibration-framework_inputs/
# with the layout documented in docs/input-layout.md.
# No input path below refers to the original author's PC.
# Prefer an explicit project root. Without it, support both launching Jupyter
# from the repository root and opening this notebook from its notebooks/ folder.
_requested_root = os.environ.get("WSIMOD_STAGED_CALIBRATION_FRAMEWORK_ROOT")
if _requested_root:
    PROJECT_ROOT = Path(_requested_root).resolve()
else:
    _cwd = Path.cwd().resolve()
    PROJECT_ROOT = (
        _cwd
        if (_cwd / "wsimod-staged-calibration-framework_inputs").exists()
        else _cwd.parent
        if (_cwd.parent / "wsimod-staged-calibration-framework_inputs").exists()
        else _cwd
    )
INPUT_ROOT = PROJECT_ROOT / "wsimod-staged-calibration-framework_inputs"
CODE_ROOT = INPUT_ROOT / "code"
MODEL_ROOT = INPUT_ROOT / "model_node_9056"
BASELINE_ROOT = INPUT_ROOT / "baseline"
STATIC_DATA_ROOT = INPUT_ROOT / "static_data"
OBSERVATION_ROOT = INPUT_ROOT / "observations"
MAPPING_ROOT = INPUT_ROOT / "mappings"
WWTW_ROOT = INPUT_ROOT / "wwtw"
ORIGINAL_WSI_ROOT = INPUT_ROOT / "wsimod_source"
WORKFLOW_ROOT = PROJECT_ROOT / "wsimod-staged-calibration-framework_outputs"
INPUT_MANIFEST = INPUT_ROOT / "wsimod-staged-calibration-framework_input_manifest.json"

FRAMEWORK_IMPLEMENTATION_VERSION = "2.3.0-framework-aligned-phaseAB-portable"
ROBIN_ORIGINAL_CONFIG = MODEL_ROOT / "config.yml"
ROUTING_BASELINE_CONFIG = BASELINE_ROOT / "corrected_wwtw_routing_baseline.yml"
ROUTING_CHANGE_LOG = BASELINE_ROOT / "confirmed_wwtw_routing_change_log.csv"
REGIONAL_GENERATOR = CODE_ROOT / "node9056_regionalisation_generator_v1_1_0.py"
REGIONAL_GENERATOR_BASE = CODE_ROOT / "node9056_regionalisation_generator_v1_0_0.py"
REGIONAL_RULES = CODE_ROOT / "regionalisation_rules_v1_1_0.yml"
PREPROCESSING_REGIONALISATION_DIR = WORKFLOW_ROOT / "preprocessing_regionalisation_v1_1_0"
REGIONALISED_RULE_CONFIG_DIR = PREPROCESSING_REGIONALISATION_DIR / "configs"
REGIONAL_GENERATOR_MANIFEST = PREPROCESSING_REGIONALISATION_DIR / "GENERATOR_MANIFEST.json"

# Workers use the Python executable of the current Jupyter kernel. Install
# WSIMOD and the listed runtime packages in that environment before execution.
WSIMOD_PYTHON = Path(sys.executable)

# All portable input aliases are declared once here and checked in Step 1.
FLOW_OBS_DIR = OBSERVATION_ROOT / "flow"
NITRATE_OBS_DIR = OBSERVATION_ROOT / "nitrate"
ALTERNATIVE_MAPPING_TABLE = MAPPING_ROOT / "official_station_arc_mapping.csv"
BARNEY_OUTLET_REFERENCE = MAPPING_ROOT / "catchment_outlets.csv"
WWTW_MANIFEST = WWTW_ROOT / "candidate_wwtw_nitrate_parameter_manifest.csv"
ARC_INDEX = MODEL_ROOT / "visualisation_node_9056" / "model_arcs_index.csv"
CATCHMENT_INDEX = MODEL_ROOT / "visualisation_node_9056" / "catchment_model_run_index.csv"
SOIL_FRACTIONS = STATIC_DATA_ROOT / "soilscapes_fractions.csv"
HYDRO_FRACTIONS = STATIC_DATA_ROOT / "hydrogeology_fractions.csv"
UNIFIED_DATA = MODEL_ROOT / "unified_data.parquet"
SELECTION_MAPPING_MODE = "official_station_alternative"

COARSE_WARMUP_START = "2009-01-01"
COARSE_WARMUP_END = "2013-12-31"
FINE_WARMUP_START = "2009-01-01"
FINE_WARMUP_END = "2013-12-31"
OUTPUT_START = "2014-01-01"
OUTPUT_END = "2019-12-31"
CALIBRATION_START = "2014-01-01"
CALIBRATION_END = "2017-12-31"
VALIDATION_START = "2018-01-01"
VALIDATION_END = "2019-12-31"
VALIDATION_CHUNK_DAYS = 365

RUN_PHASE_A = True
RUN_PHASE_B = True
FORCE_RERUN = False
MAX_PARALLEL_WORKERS = 12

# A positive delta must exceed numerical equality with the baseline.
MIN_WFD_POSITIVE_DELTA = 1e-6

# Phase-A and Phase-B mean gates. Per-WFD deltas are reported descriptively in Section 2.7.
FLOW_MEAN_DELTA_FLOOR = MIN_WFD_POSITIVE_DELTA
NITRATE_MEAN_DELTA_FLOOR = MIN_WFD_POSITIVE_DELTA
HYDRO_SCREEN_NITRATE_DELTA_FLOOR = -0.05
WQ_SCREEN_FLOW_DELTA_FLOOR = -0.01
JOINT_FLOW_WEIGHT = 0.50
JOINT_NITRATE_WEIGHT = 0.50

# H is normalised to the complete Phase-A endpoint at 1.75.
HYDROLOGY_ENDPOINT_STRENGTH = 1.75
PHASE_B_H_WEIGHTS = [0.0, 0.5, 1.0, 1.5, 1.75]
PHASE_B_W_WEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0]
PHASE_B_N_WEIGHTS = [0.0, 0.5, 1.0, 1.5]

# Multi-level Phase-A screening removes endpoint-selection bias.
PHASE_A_H_SCREEN_LEVELS = [0.5, 1.0, 1.5, 1.75]
PHASE_A_W_SCREEN_LEVELS = [0.5, 1.0, 1.5, 2.0]
PHASE_A_N_SCREEN_LEVELS = [0.5, 1.0, 1.5]

# A small tolerance classifies descriptive catchment-level response direction in Section 2.7.


## 1. Imports, paths and immutable-source contract

**Dissertation correspondence:** Section 2.1 (corrected national baseline) and the reproducibility requirements underlying Sections 2.2–2.7.

**What this step does:** Loads dependencies and freezes the model, routing and observation inputs used by all experiments

**Checks / outputs:** A source-hash mismatch stops execution before scenarios are generated


In [ ]:
# Framework correspondence: Section 2.1.
# What this cell does (Pre-experiment contract): Imports dependencies and verifies that corrected baseline inputs have not changed.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

from __future__ import annotations

import copy
import hashlib
import itertools
import json
import math
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict, deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# Fail before importing project code or generating any output if the portable
# input package is incomplete. This gives a reproducer one actionable list.
portable_input_paths = [
    INPUT_MANIFEST, ROBIN_ORIGINAL_CONFIG, ROUTING_BASELINE_CONFIG,
    ROUTING_CHANGE_LOG, REGIONAL_GENERATOR, REGIONAL_GENERATOR_BASE,
    REGIONAL_RULES, FLOW_OBS_DIR, NITRATE_OBS_DIR,
    ALTERNATIVE_MAPPING_TABLE, BARNEY_OUTLET_REFERENCE, WWTW_MANIFEST,
    ARC_INDEX, CATCHMENT_INDEX, SOIL_FRACTIONS, HYDRO_FRACTIONS,
    UNIFIED_DATA, ORIGINAL_WSI_ROOT, WSIMOD_PYTHON,
]
portable_missing = [str(path) for path in portable_input_paths if not path.exists()]
if portable_missing:
    raise FileNotFoundError(
        "Portable input check failed. Review docs/input-layout.md and provide these paths:\n"
        + "\n".join(portable_missing)
    )

FRAMEWORK_CODE_ROOT = CODE_ROOT
if str(FRAMEWORK_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_CODE_ROOT))
from node9056_regionalisation_generator_v1_1_0 import thresholded_multiplier as preprocessing_thresholded_multiplier

CONFIG_DIR = WORKFLOW_ROOT / "generated_configs"
SPEC_DIR = WORKFLOW_ROOT / "scenario_specs"
RUN_DIR = WORKFLOW_ROOT / "scenario_runs"
TABLE_DIR = WORKFLOW_ROOT / "tables"
FIGURE_DIR = WORKFLOW_ROOT / "figures"
for path in [WORKFLOW_ROOT, CONFIG_DIR, SPEC_DIR, RUN_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Portable input aliases are declared in Step 1 and validated below.

preprocessing_command = [str(WSIMOD_PYTHON), str(REGIONAL_GENERATOR), "--base-config", str(ROUTING_BASELINE_CONFIG), "--soil-fractions", str(SOIL_FRACTIONS), "--hydro-fractions", str(HYDRO_FRACTIONS), "--catchment-index", str(CATCHMENT_INDEX), "--rules", str(REGIONAL_RULES), "--output-dir", str(PREPROCESSING_REGIONALISATION_DIR)]
preprocessing_completed = subprocess.run(preprocessing_command, text=True, capture_output=True)
if preprocessing_completed.returncode != 0:
    raise RuntimeError("pre-processing regionalisation generator failed:\n" + preprocessing_completed.stderr[-4000:])
if not REGIONAL_GENERATOR_MANIFEST.exists():
    raise RuntimeError("pre-processing generator did not write its manifest")
preprocessing_manifest = json.loads(REGIONAL_GENERATOR_MANIFEST.read_text(encoding="utf-8"))
if preprocessing_manifest.get("generator_version") != "1.1.0" or preprocessing_manifest.get("weighting_mode") != "three_stage_functional_group_threshold":
    raise RuntimeError("Unexpected pre-processing generator contract")
if not preprocessing_manifest.get("physical_constraint_engine", {}).get("all_passed", False):
    raise RuntimeError("pre-processing physical-constraint gate failed")
REGIONALISED_RULE_PATHS = {
    "R01": REGIONALISED_RULE_CONFIG_DIR / "R01_soil_type_regionalised_storage.yml",
    "R02": REGIONALISED_RULE_CONFIG_DIR / "R02_soil_type_regionalised_percolation.yml",
    "R03": REGIONALISED_RULE_CONFIG_DIR / "R03_hydrogeology_regionalised_groundwater_capacity.yml",
    "R04": REGIONALISED_RULE_CONFIG_DIR / "R04_soil_storage_plus_percolation.yml",
    "R05": REGIONALISED_RULE_CONFIG_DIR / "R05_storage_percolation_groundwater_capacity.yml",
    "R06": REGIONALISED_RULE_CONFIG_DIR / "R06_hydrogeology_regionalised_residence_time.yml",
    "R07": REGIONALISED_RULE_CONFIG_DIR / "R07_full_framework_plus_residence_time.yml",
}

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

immutable_sources = [
    ROBIN_ORIGINAL_CONFIG,
    ROUTING_BASELINE_CONFIG,
    ROUTING_CHANGE_LOG,
    ALTERNATIVE_MAPPING_TABLE,
    BARNEY_OUTLET_REFERENCE,
    WWTW_MANIFEST,
    REGIONAL_GENERATOR, REGIONAL_GENERATOR_BASE, REGIONAL_RULES, SOIL_FRACTIONS, HYDRO_FRACTIONS, CATCHMENT_INDEX,
    *REGIONALISED_RULE_PATHS.values(),
    UNIFIED_DATA,
    ORIGINAL_WSI_ROOT / "wsimod" / "orchestration" / "model.py",
    ORIGINAL_WSI_ROOT / "wsimod" / "nodes" / "land.py",
    ORIGINAL_WSI_ROOT / "wsimod" / "nodes" / "nutrient_pool.py",
    ORIGINAL_WSI_ROOT / "wsimod" / "nodes" / "wtw.py",
]
required = immutable_sources + [INPUT_MANIFEST, WSIMOD_PYTHON, FLOW_OBS_DIR, NITRATE_OBS_DIR, ARC_INDEX, CATCHMENT_INDEX]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Portable input check failed. Review docs/input-layout.md and provide these paths:\n" + "\n".join(missing))

SOURCE_CONTRACT = {str(path): sha256(path) for path in immutable_sources}
(WORKFLOW_ROOT / "immutable_source_contract.json").write_text(
    json.dumps(SOURCE_CONTRACT, indent=2), encoding="utf-8"
)

def assert_sources_unchanged() -> None:
    now = {str(path): sha256(path) for path in immutable_sources}
    changed = [path for path in SOURCE_CONTRACT if now[path] != SOURCE_CONTRACT[path]]
    if changed:
        raise RuntimeError("Immutable Robin source/config changed during workflow: " + "; ".join(changed))

# The corrected baseline is permitted to differ from Robin's original config
# only in WWTW discharge arcs. Nodes, parameters and all non-WWTW arcs must match.
robin_original_config = yaml.safe_load(ROBIN_ORIGINAL_CONFIG.read_text(encoding="utf-8"))
routing_baseline_config = yaml.safe_load(ROUTING_BASELINE_CONFIG.read_text(encoding="utf-8"))
if robin_original_config["nodes"] != routing_baseline_config["nodes"]:
    raise RuntimeError("Corrected WWTW-routing baseline unexpectedly changes nodes or node parameters")
for key in set(robin_original_config) - {"arcs", "nodes"}:
    if robin_original_config.get(key) != routing_baseline_config.get(key):
        raise RuntimeError(f"Corrected WWTW-routing baseline unexpectedly changes top-level key: {key}")

def is_wwtw_discharge_arc(arc: dict) -> bool:
    return str(arc.get("in_port", "")).endswith("-wwtw")

original_non_wwtw_arcs = {
    name: arc for name, arc in robin_original_config["arcs"].items()
    if not is_wwtw_discharge_arc(arc)
}
corrected_non_wwtw_arcs = {
    name: arc for name, arc in routing_baseline_config["arcs"].items()
    if not is_wwtw_discharge_arc(arc)
}
if original_non_wwtw_arcs != corrected_non_wwtw_arcs:
    raise RuntimeError("A non-WWTW arc changed; river/network topology is not locked")

original_wwtw = {
    str(arc["in_port"]): str(arc["out_port"])
    for arc in robin_original_config["arcs"].values() if is_wwtw_discharge_arc(arc)
}
corrected_wwtw = {
    str(arc["in_port"]): str(arc["out_port"])
    for arc in routing_baseline_config["arcs"].values() if is_wwtw_discharge_arc(arc)
}
if set(original_wwtw) != set(corrected_wwtw) or not original_wwtw:
    raise RuntimeError("WWTW routing inventory changed unexpectedly")
wwtw_routing_diff = pd.DataFrame([
    {
        "wwtw_node": node,
        "original_receiving_node": original_wwtw[node],
        "corrected_receiving_node": corrected_wwtw[node],
        "changed": original_wwtw[node] != corrected_wwtw[node],
    }
    for node in sorted(original_wwtw)
])
if int(wwtw_routing_diff["changed"].sum()) != 10:
    raise RuntimeError(
        f"Expected 10 confirmed WWTW receiving-node changes relative to Robin original; "
        f"found {int(wwtw_routing_diff['changed'].sum())}"
    )
wwtw_routing_diff.to_csv(TABLE_DIR / "00_confirmed_WWTW_routing_baseline_vs_Robin_original.csv", index=False)

# Validate every receiving-node decision recorded in the routing log.
routing_log = pd.read_csv(ROUTING_CHANGE_LOG, dtype=str)
required_routing_columns = {"identifier", "after_out_port", "expected_out_port"}
if not required_routing_columns.issubset(routing_log.columns):
    raise RuntimeError(f"Routing change log is missing columns: {sorted(required_routing_columns - set(routing_log.columns))}")
routing_log["wwtw_node"] = routing_log["identifier"].astype(str) + "-wwtw"
routing_log["configured_out_port"] = routing_log["wwtw_node"].map(corrected_wwtw)
routing_log["decision_matches_config"] = (
    routing_log["configured_out_port"].eq(routing_log["after_out_port"])
    & routing_log["configured_out_port"].eq(routing_log["expected_out_port"])
)
if not routing_log["decision_matches_config"].all():
    raise RuntimeError("Corrected routing baseline does not match one or more logged Robin/user decisions")
routing_log.to_csv(TABLE_DIR / "00_confirmed_WWTW_routing_change_log_audit.csv", index=False)

# Load R00-R07 as explicit hydrology rule candidates. R00 is the unregionalised
# parameter baseline; every R01-R07 candidate receives the same confirmed routing arcs.
regionalised_source_configs = {"R00": robin_original_config}
regionalised_source_configs.update({
    rule_id: yaml.safe_load(path.read_text(encoding="utf-8"))
    for rule_id, path in REGIONALISED_RULE_PATHS.items()
})

def leaf_differences(left, right, path=()):
    if isinstance(left, dict) and isinstance(right, dict):
        for key in sorted(set(left) | set(right)):
            if key not in left or key not in right:
                yield path + (str(key),), left.get(key), right.get(key)
            else:
                yield from leaf_differences(left[key], right[key], path + (str(key),))
    elif isinstance(left, list) and isinstance(right, list):
        if len(left) != len(right):
            yield path + ("length",), len(left), len(right)
        for index, (old, new) in enumerate(zip(left, right)):
            yield from leaf_differences(old, new, path + (str(index),))
    elif left != right:
        yield path, left, right

allowed_regionalised_keys = {
    "Land": {"wilting_point", "field_capacity", "total_porosity", "percolation_coefficient"},
    "Groundwater": {"capacity", "initial_storage", "residence_time"},
}
regionalised_rule_audit_rows = []
composite_rule_configs = {}
for rule_id, source_cfg in regionalised_source_configs.items():
    for key in set(robin_original_config) - {"arcs", "nodes"}:
        if source_cfg.get(key) != robin_original_config.get(key):
            raise RuntimeError(f"{rule_id} changes locked top-level key {key}")
    source_non_wwtw = {name: arc for name, arc in source_cfg["arcs"].items() if not is_wwtw_discharge_arc(arc)}
    if source_non_wwtw != original_non_wwtw_arcs:
        raise RuntimeError(f"{rule_id} changes locked non-WWTW topology")
    node_diffs = list(leaf_differences(robin_original_config["nodes"], source_cfg["nodes"]))
    for path, old, new in node_diffs:
        node_name = path[0]
        node_type = robin_original_config["nodes"][node_name].get("type_")
        allowed = node_type in allowed_regionalised_keys and path[-1] in allowed_regionalised_keys[node_type]
        regionalised_rule_audit_rows.append({
            "rule_id": rule_id, "node": node_name, "node_type": node_type,
            "path": "/".join(path), "parameter": path[-1], "old_value": old, "new_value": new, "allowed": allowed,
        })
        if not allowed:
            raise RuntimeError(f"{rule_id} contains unregistered regionalised change at {'/'.join(path)}")
    composite = copy.deepcopy(source_cfg)
    composite["arcs"] = copy.deepcopy(routing_baseline_config["arcs"])
    composite_rule_configs[rule_id] = composite

def set_nested_value(mapping, path, value):
    target = mapping
    for key in path[:-1]:
        target = target[key]
    target[path[-1]] = value

# pre-processing weighted regionalised-rule library. Every R01-R07 rule is treated
# identically. A weighted configuration is R00 + w * (Rxx - R00).
def iter_preprocessing_surface_items(node):
    surfaces = node.get("surfaces", {})
    if isinstance(surfaces, dict):
        yield from surfaces.items()
    else:
        for index, surface in enumerate(surfaces):
            yield str(surface.get("surface", index)), surface

def regionalised_config_is_physical(cfg):
    for node in cfg["nodes"].values():
        if node.get("type_") == "Land":
            if any(float(node[key]) <= 0 for key in ["surface_residence_time", "subsurface_residence_time", "percolation_residence_time"]):
                return False
            for _, surface in iter_preprocessing_surface_items(node):
                if all(key in surface for key in ["wilting_point", "field_capacity", "total_porosity"]):
                    wp, fc, tp = (float(surface[key]) for key in ["wilting_point", "field_capacity", "total_porosity"])
                    if not (0 < wp < fc < tp <= 1):
                        return False
                for key in ["surface_coefficient", "percolation_coefficient"]:
                    if key in surface and not (0 <= float(surface[key]) <= 1):
                        return False
        elif node.get("type_") == "Groundwater":
            capacity = float(node["capacity"])
            initial = float(node["initial_storage"])
            residence = float(node["residence_time"])
            if not (capacity > 0 and 0 <= initial <= capacity and residence > 0):
                return False
    return True

REGIONALISED_CANDIDATE_BY_INDEX = {0: "R00"}
REGIONALISED_WEIGHT_INDEX = {}
next_regionalised_index = 1
for rule_number in range(1, 8):
    rule_id = f"R{rule_number:02d}"
    REGIONALISED_CANDIDATE_BY_INDEX[next_regionalised_index] = rule_id
    REGIONALISED_WEIGHT_INDEX[(rule_id, 1.0)] = next_regionalised_index
    next_regionalised_index += 1

weighted_rule_exclusions = []
for rule_number in range(1, 8):
    rule_id = f"R{rule_number:02d}"
    for weight in [0.5, 1.5, 1.75]:
        candidate_id = f"{rule_id}W{int(weight * 100):03d}"
        weighted = copy.deepcopy(composite_rule_configs["R00"])
        candidate_audit_rows = []
        for path, old, rule_value in leaf_differences(composite_rule_configs["R00"]["nodes"], composite_rule_configs[rule_id]["nodes"]):
            new_value = float(old) + weight * (float(rule_value) - float(old))
            set_nested_value(weighted["nodes"], path, new_value)
            node_name = path[0]
            node_type = robin_original_config["nodes"][node_name].get("type_")
            candidate_audit_rows.append({
                "rule_id": candidate_id, "node": node_name, "node_type": node_type,
                "path": "/".join(path), "parameter": path[-1], "old_value": old,
                "new_value": new_value, "allowed": True, "anchor": rule_id,
                "delta_weight": weight,
            })
        weighted["arcs"] = copy.deepcopy(routing_baseline_config["arcs"])
        if regionalised_config_is_physical(weighted):
            composite_rule_configs[candidate_id] = weighted
            regionalised_rule_audit_rows.extend(candidate_audit_rows)
            REGIONALISED_CANDIDATE_BY_INDEX[next_regionalised_index] = candidate_id
            REGIONALISED_WEIGHT_INDEX[(rule_id, weight)] = next_regionalised_index
            next_regionalised_index += 1
        else:
            weighted_rule_exclusions.append({"anchor_rule": rule_id, "weight": weight, "reason": "physical constraint violation"})

pd.DataFrame(weighted_rule_exclusions, columns=["anchor_rule", "weight", "reason"]).to_csv(
    TABLE_DIR / "00_excluded_weighted_regionalised_rules.csv", index=False
)

regionalised_rule_audit = pd.DataFrame(regionalised_rule_audit_rows)
regionalised_rule_audit.to_csv(TABLE_DIR / "00_R00_R07_regionalised_rule_leaf_change_audit.csv", index=False)
regionalised_rule_summary = pd.DataFrame([
    {"rule_id": rule_id, "changed_leaf_parameters": sum(row["rule_id"] == rule_id for row in regionalised_rule_audit_rows)}
    for rule_id in composite_rule_configs
])
regionalised_rule_summary.to_csv(TABLE_DIR / "00_R00_R07_regionalised_rule_summary.csv", index=False)

dates = pd.DatetimeIndex(sorted(pd.to_datetime(pd.read_parquet(UNIFIED_DATA, columns=["time"])["time"].unique())))
expected_coarse_warmup = pd.date_range(COARSE_WARMUP_START, COARSE_WARMUP_END, freq="D")
expected_fine_warmup = pd.date_range(FINE_WARMUP_START, FINE_WARMUP_END, freq="D")
expected_output = pd.date_range(OUTPUT_START, OUTPUT_END, freq="D")
if not expected_coarse_warmup.isin(dates).all() or not expected_fine_warmup.isin(dates).all() or not expected_output.isin(dates).all():
    raise RuntimeError("Unified data does not cover the complete warm-up/output contract")
if len(expected_coarse_warmup) != 1826 or len(expected_fine_warmup) != 1826 or not expected_coarse_warmup.equals(expected_fine_warmup):
    raise RuntimeError(f"Every stage must use the identical 1,826-day warm-up: coarse={len(expected_coarse_warmup)}, fine={len(expected_fine_warmup)}")

print("Workflow:", WORKFLOW_ROOT)
print("Robin original config SHA256:", SOURCE_CONTRACT[str(ROBIN_ORIGINAL_CONFIG)])
print("Corrected WWTW-routing baseline SHA256:", SOURCE_CONTRACT[str(ROUTING_BASELINE_CONFIG)])
print("Regionalised hydrology rules:", sorted(composite_rule_configs))
print("Phase A/B warm-up:", len(expected_coarse_warmup), COARSE_WARMUP_START, "to", COARSE_WARMUP_END)
print("Shortlist/fine warm-up:", len(expected_fine_warmup), FINE_WARMUP_START, "to", FINE_WARMUP_END)
print("Output days:", len(expected_output), OUTPUT_START, "to", OUTPUT_END)
print("Confirmed WWTW receiving-node changes:", int(wwtw_routing_diff["changed"].sum()))
print("Non-WWTW arcs locked:", len(corrected_non_wwtw_arcs))


## 2. Observation-to-output mapping

**Dissertation correspondence:** Sections 2.1 and 2.2.

**What this step does:** Builds the confirmed station-to-Arc matrix and the recorded-Arc union required for common evaluation

**Checks / outputs:** Exports auditable mapping tables before any KGE calculation


In [ ]:
# Framework correspondence: Sections 2.1 and 2.2.
# What this cell does (Observation-to-output mapping): Builds auditable observation/Arc matrices before any performance calculation.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

model_catchments = set(
    pd.read_csv(CATCHMENT_INDEX)["wb_id"].dropna().astype(str)
)
model_arcs = set(pd.read_csv(ARC_INDEX)["arc"].dropna().astype(str))

def original_observation_matrix() -> pd.DataFrame:
    rows = []
    for dataset, folder in [("flow", FLOW_OBS_DIR), ("nitrate", NITRATE_OBS_DIR)]:
        assigned = pd.read_csv(folder / "assigned_information.csv", dtype={"catchment_id": str})
        assigned = assigned[assigned["catchment_id"].isin(model_catchments)].copy()
        assigned["dataset"] = dataset
        assigned["senior_file"] = assigned["catchment_id"].map(lambda x: str(folder / f"{x}_sim_obs.csv"))
        assigned["mapping_status"] = "Robin original assigned_information.csv"
        rows.append(assigned[["dataset", "catchment_id", "outfall_arc", "senior_file", "mapping_status"]])
    return pd.concat(rows, ignore_index=True)

def alternative_observation_matrix() -> pd.DataFrame:
    mapping = pd.read_csv(ALTERNATIVE_MAPPING_TABLE, dtype={"wfd_id": str})
    mapping = mapping.rename(columns={"wfd_id": "catchment_id", "proposed_observation_arc": "outfall_arc"})
    mapping["senior_file"] = mapping.apply(
        lambda row: str((FLOW_OBS_DIR if row["dataset"] == "flow" else NITRATE_OBS_DIR) / f"{row['catchment_id']}_sim_obs.csv"),
        axis=1,
    )
    mapping["mapping_status"] = "Barney-confirmed official-station Arc mapping"
    return mapping[["dataset", "catchment_id", "outfall_arc", "senior_file", "mapping_status"]]

if SELECTION_MAPPING_MODE not in {"robin_original", "official_station_alternative"}:
    raise ValueError("Invalid SELECTION_MAPPING_MODE")

observation_matrices = {
    "robin_original": original_observation_matrix(),
    "official_station_alternative": alternative_observation_matrix(),
}

def normalise_barney_edge(edge: str) -> str:
    if pd.isna(edge):
        return np.nan
    edge = str(edge).strip()
    if not edge or "-" not in edge:
        return np.nan
    if edge.endswith(".reversed"):
        edge = edge[:-len(".reversed")]
    left, right = edge.split("-", 1)
    return f"{left}-to-{right}"

barney_reference = pd.read_csv(BARNEY_OUTLET_REFERENCE, dtype={"wb_id": str})
barney_reference = barney_reference[["wb_id", "catchment_outlet_edge_id"]].rename(columns={"wb_id": "catchment_id"})
barney_reference["barney_confirmed_arc"] = barney_reference["catchment_outlet_edge_id"].map(normalise_barney_edge)
confirmed_mapping_audit = observation_matrices["official_station_alternative"].merge(
    barney_reference[["catchment_id", "barney_confirmed_arc"]],
    on="catchment_id", how="left", validate="many_to_one",
)
confirmed_mapping_audit["matches_Barney"] = confirmed_mapping_audit["outfall_arc"].eq(confirmed_mapping_audit["barney_confirmed_arc"])
if confirmed_mapping_audit["barney_confirmed_arc"].isna().any() or not confirmed_mapping_audit["matches_Barney"].all():
    raise RuntimeError("Official-station mapping does not match Barney's confirmed outlet reference")
confirmed_mapping_audit.to_csv(TABLE_DIR / "00_confirmed_observation_mapping_vs_Barney_audit.csv", index=False)

matrix_hashes = {}
clean_matrices = {}
for mapping_mode, matrix in observation_matrices.items():
    matrix = matrix.copy()
    matrix["mapping_mode"] = mapping_mode
    matrix["arc_exists"] = matrix["outfall_arc"].isin(model_arcs)
    matrix["observation_file_exists"] = matrix["senior_file"].map(lambda x: Path(x).exists())
    matrix = matrix[matrix["arc_exists"] & matrix["observation_file_exists"]].copy()
    if matrix.duplicated(["dataset", "catchment_id"]).any() or matrix.empty:
        raise RuntimeError(f"Invalid observation matrix: {mapping_mode}")
    path = TABLE_DIR / f"00_observation_matrix__{mapping_mode}.csv"
    matrix.to_csv(path, index=False)
    matrix_hashes[mapping_mode] = sha256(path)
    clean_matrices[mapping_mode] = matrix
observation_matrices = clean_matrices
observation_matrix_bundle = pd.concat(observation_matrices.values(), ignore_index=True)
OBS_MATRIX_BUNDLE_PATH = TABLE_DIR / "00_observation_matrix_bundle.csv"
observation_matrix_bundle.to_csv(OBS_MATRIX_BUNDLE_PATH, index=False)
OBS_MATRIX_BUNDLE_SHA256 = sha256(OBS_MATRIX_BUNDLE_PATH)
record_arcs = sorted(observation_matrix_bundle["outfall_arc"].unique())
pd.DataFrame({"record_arc": record_arcs}).to_csv(TABLE_DIR / "00_record_arcs_union_both_mappings.csv", index=False)
selection_matrix = observation_matrices[SELECTION_MAPPING_MODE]
flow_wfds = set(selection_matrix.loc[selection_matrix["dataset"].eq("flow"), "catchment_id"].astype(str))
nitrate_wfds = set(selection_matrix.loc[selection_matrix["dataset"].eq("nitrate"), "catchment_id"].astype(str))
PRIMARY_PAIRED_WFDS = sorted(flow_wfds & nitrate_wfds)
if len(PRIMARY_PAIRED_WFDS) != 9:
    raise RuntimeError(f"Expected exactly 9 paired Q-WQ WFDs; found {PRIMARY_PAIRED_WFDS}")
pd.DataFrame({"catchment_id":PRIMARY_PAIRED_WFDS,"selection_scope":"primary_paired_Q_WQ"}).to_csv(TABLE_DIR/"00_primary_9_paired_WFDs.csv",index=False)

paired_mapping = observation_matrices["robin_original"][["dataset", "catchment_id", "outfall_arc"]].rename(columns={"outfall_arc": "original_arc"}).merge(
    observation_matrices["official_station_alternative"][["dataset", "catchment_id", "outfall_arc"]].rename(columns={"outfall_arc": "alternative_arc"}),
    on=["dataset", "catchment_id"], how="outer",
)
paired_mapping["arc_changed"] = paired_mapping["original_arc"] != paired_mapping["alternative_arc"]
paired_mapping.to_csv(TABLE_DIR / "00_original_vs_alternative_mapping_crosswalk.csv", index=False)
print("Parameter-selection mapping:", SELECTION_MAPPING_MODE)
print("Original series:", len(observation_matrices["robin_original"]))
print("Confirmed official-station series:", len(observation_matrices["official_station_alternative"]))
print("Union record arcs:", len(record_arcs))
print("Mapping bundle SHA256:", OBS_MATRIX_BUNDLE_SHA256)
display(paired_mapping)


## 3. Evidence-informed parameter registry and pre-processing regionalisation

**Dissertation correspondence:** Sections 2.3.1 and 2.4.

**What this step does:** Defines functional parameter groups and calculates regionalisation factors from Soilscapes, hydrogeology and land cover without using model performance

**Checks / outputs:** Writes regionalisation and rule-contract audits


In [ ]:
# Framework correspondence: Sections 2.3.1 and 2.4.
# What this cell does (Evidence-informed parameter organisation): Constructs fixed spatial regionalisation factors without using KGE.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

SURFACE_GROUPS = {
    "grass": ["Grass"],
    "garden": ["Garden"],
    "woody_natural": ["Trees and Scrubs, short Woody plants, hedgerows", "Heather", "Heathland and Bracken"],
    "cereals": ["Winter Wheat", "Spring Wheat", "Winter Barley", "Spring Barley", "Winter Oats", "Spring Oats", "Winter Rye", "Maize"],
    "oilseed_legume": ["Winter Oilseed", "Spring Linseed", "Lucerne", "Winter Field beans", "Spring Field beans", "Spring Peas", "Clover"],
    "roots_vegetables_fallow": ["Beet", "Potato", "Onions", "Lettuce", "Fallow Land"],
}
SURFACE_TO_GROUP = {surface: group for group, members in SURFACE_GROUPS.items() for surface in members}
SOIL_N_PRIOR_MULTIPLIERS = {group: 1.0 for group in SURFACE_GROUPS}
SOIL_N_PRIOR_MULTIPLIERS.update({"garden": 2.0, "roots_vegetables_fallow": 2.0})

def iter_config_surface_items(node):
    surfaces=node.get("surfaces",{})
    if isinstance(surfaces,dict):
        yield from surfaces.items()
    else:
        for index,surface in enumerate(surfaces):
            yield str(surface.get("surface",index)),surface

surface_inventory_rows=[]
for node_name,node in robin_original_config["nodes"].items():
    if node.get("type_") != "Land":
        continue
    for surface_key,surface in iter_config_surface_items(node):
        if surface.get("type_") == "ImperviousSurface":
            continue
        surface_name=str(surface.get("surface",surface_key))
        surface_inventory_rows.append({"node":node_name,"surface":surface_name,"surface_group":SURFACE_TO_GROUP.get(surface_name),"area":float(surface.get("area",0.0))})
surface_inventory=pd.DataFrame(surface_inventory_rows)
if len(surface_inventory) != 827 or surface_inventory["surface_group"].isna().any() or len(SURFACE_TO_GROUP) != sum(len(v) for v in SURFACE_GROUPS.values()):
    raise RuntimeError("Land-cover soil-N groups must cover all 827 nutrient-pool surfaces exactly once")
surface_inventory["catchment_area_total"]=surface_inventory.groupby("node")["area"].transform("sum")
surface_inventory["catchment_landcover_fraction"]=surface_inventory["area"]/surface_inventory["catchment_area_total"]
surface_inventory.to_csv(TABLE_DIR/"01_landcover_surface_inventory_and_fraction.csv",index=False)
SOIL_N_MIXTURE_THRESHOLD = 0.60
SOIL_N_DOMINANT_THRESHOLD = 0.80
SOIL_N_REFERENCE_MINFPAR = 0.00013

node_group_area = surface_inventory.groupby(["node", "surface_group"], as_index=False)["area"].sum()
node_group_fraction = node_group_area.pivot(index="node", columns="surface_group", values="area").fillna(0.0)
node_group_fraction = node_group_fraction.reindex(columns=list(SURFACE_GROUPS), fill_value=0.0)
node_group_fraction = node_group_fraction.div(node_group_fraction.sum(axis=1), axis=0)
if not np.allclose(node_group_fraction.sum(axis=1), 1.0, atol=1e-12):
    raise RuntimeError("Soil-N land-cover fractions do not sum to one within each Land node")

# Each land-cover class is one functional group. This is intentionally the
# same thresholded_multiplier implementation used by pre-processing Soilscapes/BGS.
SOIL_N_FUNCTIONAL_GROUPS = {group: [group] for group in SURFACE_GROUPS}
soil_n_coverage = pd.Series(1.0, index=node_group_fraction.index)
soil_n_preprocessing_factor, soil_n_preprocessing_audit = preprocessing_thresholded_multiplier(
    fractions=node_group_fraction,
    columns=list(SURFACE_GROUPS),
    multiplier_by_column=SOIL_N_PRIOR_MULTIPLIERS,
    coverage=soil_n_coverage,
    uncovered_multiplier=1.0,
    functional_groups=SOIL_N_FUNCTIONAL_GROUPS,
    uncovered_group="unclassified_or_water",
    mixture_threshold=SOIL_N_MIXTURE_THRESHOLD,
    dominant_threshold=SOIL_N_DOMINANT_THRESHOLD,
)
if (soil_n_preprocessing_factor <= 0).any():
    raise RuntimeError("Every Soil-N pre-processing regionalisation factor must be positive")

SOIL_N_NODE_FRACTIONS = {
    str(node): {group: float(node_group_fraction.loc[node, group]) for group in SURFACE_GROUPS}
    for node in node_group_fraction.index
}
SOIL_N_PREPROCESSING_FACTOR_BY_NODE = {
    str(node): float(soil_n_preprocessing_factor.loc[node])
    for node in node_group_fraction.index
}
SOIL_N_NODE_CLASSIFICATION = {
    str(node): {
        "dominant_group": str(soil_n_preprocessing_audit.loc[node, "dominant_functional_group"]),
        "dominant_share": float(soil_n_preprocessing_audit.loc[node, "dominant_functional_group_share"]),
        "threshold_stage": str(soil_n_preprocessing_audit.loc[node, "threshold_stage"]),
        "dominant_blend_alpha": float(soil_n_preprocessing_audit.loc[node, "dominant_blend_alpha"]),
        "mixture_multiplier": float(soil_n_preprocessing_audit.loc[node, "mixture_multiplier"]),
        "dominant_multiplier": float(soil_n_preprocessing_audit.loc[node, "dominant_multiplier"]),
        "preprocessing_regionalisation_factor": float(soil_n_preprocessing_factor.loc[node]),
    }
    for node in node_group_fraction.index
}

soil_n_preprocessing_table = soil_n_preprocessing_audit.copy()
soil_n_preprocessing_table.insert(0, "node", soil_n_preprocessing_table.index.astype(str))
soil_n_preprocessing_table["preprocessing_minfpar_N"] = SOIL_N_REFERENCE_MINFPAR * soil_n_preprocessing_factor.to_numpy()
for group in SURFACE_GROUPS:
    soil_n_preprocessing_table[f"q__{group}"] = node_group_fraction[group].to_numpy()
    soil_n_preprocessing_table[f"mi__{group}"] = float(SOIL_N_PRIOR_MULTIPLIERS[group])
soil_n_preprocessing_table.to_csv(TABLE_DIR / "01_soil_N_preprocessing_regionalisation.csv", index=False)

surface_group_summary = surface_inventory.groupby("surface_group", as_index=False).agg(
    surface_instances=("surface", "size"),
    catchments=("node", "nunique"),
    total_area=("area", "sum"),
)
surface_group_summary["area_fraction"] = surface_group_summary["total_area"] / surface_group_summary["total_area"].sum()
surface_group_summary["regionalisation_class_multiplier_mi"] = surface_group_summary["surface_group"].map(SOIL_N_PRIOR_MULTIPLIERS)
surface_group_summary.to_csv(TABLE_DIR / "01_landcover_soil_N_group_summary.csv", index=False)

surface_inventory_with_preprocessing = surface_inventory.copy()
surface_inventory_with_preprocessing["preprocessing_regionalisation_factor"] = surface_inventory_with_preprocessing["node"].map(SOIL_N_PREPROCESSING_FACTOR_BY_NODE)
SOIL_N_PREPROCESSING_AREA_WEIGHTED_MEAN = float(np.average(
    surface_inventory_with_preprocessing["preprocessing_regionalisation_factor"],
    weights=surface_inventory_with_preprocessing["area"],
))

# WFD710 is retained as a worked audit row; the values are derived, not hard-coded.
WFD710_SOIL_N_NODE = "GB105036040710-land"
if WFD710_SOIL_N_NODE not in SOIL_N_PREPROCESSING_FACTOR_BY_NODE:
    raise RuntimeError("WFD710 Land node is missing from the Soil-N pre-processing evidence table")
wfd710_q = SOIL_N_NODE_FRACTIONS[WFD710_SOIL_N_NODE]
wfd710_M = sum(wfd710_q[group] * SOIL_N_PRIOR_MULTIPLIERS[group] for group in SURFACE_GROUPS)
wfd710_D = max(wfd710_q.values())
wfd710_a = float(np.clip(
    (wfd710_D - SOIL_N_MIXTURE_THRESHOLD) / (SOIL_N_DOMINANT_THRESHOLD - SOIL_N_MIXTURE_THRESHOLD),
    0.0, 1.0,
))
wfd710_dominant_group = max(wfd710_q, key=wfd710_q.get)
wfd710_V = float(SOIL_N_PRIOR_MULTIPLIERS[wfd710_dominant_group])
wfd710_F = (1.0 - wfd710_a) * wfd710_M + wfd710_a * wfd710_V
if not np.isclose(wfd710_F, SOIL_N_PREPROCESSING_FACTOR_BY_NODE[WFD710_SOIL_N_NODE], rtol=1e-12, atol=1e-12):
    raise RuntimeError("WFD710 manual Soil-N pre-processing calculation does not match the shared regionalisation function")
pd.DataFrame([{
    "node": WFD710_SOIL_N_NODE,
    "M": wfd710_M,
    "D": wfd710_D,
    "a": wfd710_a,
    "dominant_group": wfd710_dominant_group,
    "V": wfd710_V,
    "F_preprocessing": wfd710_F,
    "reference_minfpar_N": SOIL_N_REFERENCE_MINFPAR,
    "preprocessing_minfpar_N": SOIL_N_REFERENCE_MINFPAR * wfd710_F,
}]).to_csv(TABLE_DIR / "01_WFD710_soil_N_preprocessing_worked_values.csv", index=False)

SOIL_N_RULE_CONTRACT = {
    "rule": "per-Land-node three-stage functional-group threshold; no whole-model normalisation",
    "shared_function": "node9056_regionalisation_generator_v1_1_0.thresholded_multiplier",
    "reference_minfpar_N": SOIL_N_REFERENCE_MINFPAR,
    "nutrient_pool_surface_count": int(len(surface_inventory)),
    "surface_groups": SURFACE_GROUPS,
    "surface_to_group": SURFACE_TO_GROUP,
    "class_multipliers_mi": SOIL_N_PRIOR_MULTIPLIERS,
    "functional_groups": SOIL_N_FUNCTIONAL_GROUPS,
    "dominance_thresholds": {
        "mixture_upper_exclusive": SOIL_N_MIXTURE_THRESHOLD,
        "dominant_lower_inclusive": SOIL_N_DOMINANT_THRESHOLD,
    },
    "preprocessing_factor_by_node": SOIL_N_PREPROCESSING_FACTOR_BY_NODE,
    "preprocessing_area_weighted_mean_for_audit_only": SOIL_N_PREPROCESSING_AREA_WEIGHTED_MEAN,
    "whole_model_normalisation_applied": False,
    "denpar_scope": "global Phase-A process parameter; pre-processing value fixed at 0.015",
}
SOIL_N_RULE_CONTRACT_JSON = json.dumps(SOIL_N_RULE_CONTRACT, sort_keys=True, separators=(",", ":"))
SOIL_N_RULE_CONTRACT_SHA256 = hashlib.sha256(SOIL_N_RULE_CONTRACT_JSON.encode("utf-8")).hexdigest()
SOIL_N_RULE_CONTRACT_PATH = TABLE_DIR / "01_soil_N_preprocessing_rule_contract.json"
SOIL_N_RULE_CONTRACT_PATH.write_text(json.dumps(SOIL_N_RULE_CONTRACT, indent=2), encoding="utf-8")

PARAMETER_RULES = {
    "regionalised_rule_index": {
        "group": "hydrology", "baseline": 0.0,
        "coarse_levels": [REGIONALISED_WEIGHT_INDEX[(f"R{i:02d}", 1.0)] for i in range(1, 8)],
        "lower": 0.0, "upper": float(max(REGIONALISED_CANDIDATE_BY_INDEX)),
        "fine_levels": list(REGIONALISED_CANDIDATE_BY_INDEX), "fine_search": False,
        "target": "R01-R07 complete W175 endpoints in Phase A; complete H0.5/H1.0/H1.5/H1.75 states in Phase B",
        "meaning": "current pre-processing v1.1 categorical regionalised soil/hydrogeology field; archived R07 leaf values are not imported",
    },
    "surface_coefficient_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.6, "upper": 1.4,
        "fine_levels": [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4], "fine_search": True,
        "target": "all pervious surfaces with surface_coefficient", "meaning": "multiplier on surface runoff coefficient",
    },
    "ihacres_p_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.6, "upper": 1.4,
        "fine_levels": [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4], "fine_search": True,
        "target": "all surfaces with ihacres_p", "meaning": "multiplier on IHACRES drying/soil-moisture parameter",
    },
    "land_residence_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.5, "upper": 1.5,
        "fine_levels": [0.5, 0.625, 0.75, 0.875, 1.0, 1.125, 1.25, 1.375, 1.5], "fine_search": True,
        "target": "Land surface/subsurface/percolation residence times together", "meaning": "coherent pathway residence-time multiplier",
    },
    "percolation_coefficient_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.5, "upper": 1.25,
        "fine_levels": [0.5, 0.625, 0.75, 0.875, 1.0, 1.125, 1.25], "fine_search": True,
        "target": "all pervious surfaces with percolation_coefficient", "meaning": "multiplier on percolation coefficient",
    },
    "soil_storage_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.9, 1.1], "lower": 0.8, "upper": 1.2,
        "fine_levels": [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2], "fine_search": True,
        "target": "wilting point, field capacity and total porosity together", "meaning": "coherent soil-storage scaling preserving order",
    },
    "groundwater_residence_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.5, "upper": 1.5,
        "fine_levels": [0.5, 0.625, 0.75, 0.875, 1.0, 1.125, 1.25, 1.375, 1.5], "fine_search": True,
        "target": "all Groundwater residence_time values", "meaning": "groundwater response-time multiplier",
    },
    "groundwater_storage_multiplier": {
        "group": "hydrology", "baseline": 1.0, "coarse_levels": [0.8, 1.2], "lower": 0.5, "upper": 1.5,
        "fine_levels": [0.5, 0.625, 0.75, 0.875, 1.0, 1.125, 1.25, 1.375, 1.5], "fine_search": True,
        "target": "Groundwater capacity and initial_storage together", "meaning": "coherent groundwater-storage scaling",
    },
    "wwtw_nitrate_constant": {
        "group": "wwtw", "baseline": 20.0, "coarse_levels": [16.0, 18.0, 22.0, 24.0], "lower": 16.0, "upper": 24.0,
        "fine_levels": [16, 17, 18, 19, 20, 21, 22, 23, 24], "fine_search": True,
        "target": "all model-defined WWTWs with one globally shared value", "meaning": "global raw process_parameters.nitrate.constant; not a removal percentage",
    },
    "denpar": {
        "group": "soil_nitrate", "baseline": 0.015, "coarse_levels": [0.012, 0.018], "lower": 0.009, "upper": 0.021,
        "fine_levels": [0.009, 0.0105, 0.012, 0.0135, 0.015, 0.0165, 0.018, 0.0195, 0.021], "fine_search": True,
        "target": "all GrowingSurface subclasses exposing denpar", "meaning": "existing denitrification rate coefficient",
    },
    "minfpar_N_process_multiplier": {
        "group": "soil_nitrate", "baseline": 1.0, "coarse_levels": [0.5, 2.0], "lower": 0.25, "upper": 3.0,
        "fine_levels": [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0], "fine_search": True,
        "target": "mandatory per-node pre-processing minfpar_N field",
        "meaning": "Phase-A process intensity multiplier; pre-processing regionalisation is always retained",
    },
}
parameter_registry = pd.DataFrame([
    {"parameter": name, **rule, "coarse_levels": json.dumps(rule["coarse_levels"]), "fine_levels": json.dumps(rule["fine_levels"])}
    for name, rule in PARAMETER_RULES.items()
])
parameter_registry.to_csv(TABLE_DIR / "01_parameter_registry.csv", index=False)
display(parameter_registry)


## 4. Scenario builder and parameter-change audit

**Dissertation correspondence:** Sections 2.3–2.5.

**What this step does:** Builds each candidate from the corrected routing baseline and records all allowed parameter changes

**Checks / outputs:** Generated YAML, JSON scenario specifications and strict configuration-difference audits


In [ ]:
# Framework correspondence: Sections 2.3 and 2.4.
# What this cell does (Scenario materialisation): Starts every candidate from the corrected routing baseline and records parameter changes.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

baseline_config = composite_rule_configs["R00"]
wwtw_manifest = pd.read_csv(WWTW_MANIFEST)
evidence_wwtws = sorted(wwtw_manifest["wwtw_node"].dropna().astype(str).unique())
all_wwtws = sorted(name for name, node in baseline_config["nodes"].items() if node.get("type_") == "WWTW")
if len(evidence_wwtws) != 3 or not set(evidence_wwtws).issubset(all_wwtws):
    raise RuntimeError(f"Expected exactly three evidence-matched WWTWs, found {evidence_wwtws}")
if not all_wwtws:
    raise RuntimeError("No WWTW nodes found in the corrected baseline")
target_wwtws = list(all_wwtws)
baseline_wwtw_constants = {
    node_name: float(baseline_config["nodes"][node_name]["process_parameters"]["nitrate"]["constant"])
    for node_name in all_wwtws
}
baseline_wwtw_exponents = {
    node_name: float(baseline_config["nodes"][node_name]["process_parameters"]["nitrate"]["exponent"])
    for node_name in all_wwtws
}
if not all(np.isclose(value, 20.0) for value in baseline_wwtw_constants.values()):
    raise RuntimeError("The registered WWTW baseline is 20.0, but at least one model WWTW differs")
if not all(np.isclose(value, 1.05) for value in baseline_wwtw_exponents.values()):
    raise RuntimeError("A WWTW nitrate exponent differs from the locked 1.05 baseline")
wwtw_scope = pd.DataFrame({
    "wwtw_node": all_wwtws,
    "global_parameter_applied": True,
    "has_UWWTD_N_removal_evidence": [name in evidence_wwtws for name in all_wwtws],
    "baseline_nitrate_constant": [baseline_wwtw_constants[name] for name in all_wwtws],
    "locked_nitrate_exponent": [baseline_wwtw_exponents[name] for name in all_wwtws],
})
WWTW_SCOPE_PATH = TABLE_DIR / "01_global_WWTW_nitrate_constant_scope.csv"
wwtw_scope.to_csv(WWTW_SCOPE_PATH, index=False)
WWTW_SCOPE_SHA256 = sha256(WWTW_SCOPE_PATH)
print(f"Global WWTW constant scope: {len(target_wwtws)} model-defined WWTWs; evidence-matched subset: {len(evidence_wwtws)}")

BASE_PARAMETERS = {name: float(rule["baseline"]) for name, rule in PARAMETER_RULES.items()}
LAND_RESIDENCE_KEYS = ["surface_residence_time", "subsurface_residence_time", "percolation_residence_time"]

def iter_surface_items(node: dict):
    surfaces = node.get("surfaces", {})
    if isinstance(surfaces, dict):
        yield from surfaces.items()
    else:
        for index, surface in enumerate(surfaces):
            yield str(surface.get("surface", index)), surface

def log_change(rows, scenario_id, group, path, old, new, rule):
    if not np.isclose(float(old), float(new), rtol=1e-12, atol=1e-12):
        rows.append({
            "scenario_id": scenario_id, "group": group, "path": path,
            "old_value": float(old), "new_value": float(new), "rule": rule,
        })

def build_scenario_config(scenario_id: str, parameters: dict):
    unknown = sorted(set(parameters) - set(PARAMETER_RULES))
    if unknown:
        raise KeyError(f"Unregistered parameters: {unknown}")
    p = {**BASE_PARAMETERS, **{k: float(v) for k, v in parameters.items()}}
    for name, value in p.items():
        rule = PARAMETER_RULES[name]
        if not (float(rule["lower"]) <= value <= float(rule["upper"])):
            raise ValueError(f"{name}={value} outside [{rule['lower']}, {rule['upper']}]")

    regionalised_index = p["regionalised_rule_index"]
    if not np.isclose(regionalised_index, round(regionalised_index)):
        raise ValueError(f"regionalised_rule_index must be an integer category, found {regionalised_index}")
    regionalised_rule_id = REGIONALISED_CANDIDATE_BY_INDEX.get(int(round(regionalised_index)))
    if regionalised_rule_id not in composite_rule_configs:
        raise KeyError(f"Unknown regionalised rule index: {regionalised_index}")
    cfg = copy.deepcopy(composite_rule_configs[regionalised_rule_id])
    regionalised_base = copy.deepcopy(cfg)
    changes = []
    for path, old, new in leaf_differences(baseline_config["nodes"], cfg["nodes"]):
        log_change(changes, scenario_id, "hydrology", f"nodes/{'/'.join(path)}", old, new, f"{regionalised_rule_id} regionalised parameter-setting rule")
    for node_name, node in cfg["nodes"].items():
        if node.get("type_") == "Land":
            land_mult = p["land_residence_multiplier"]
            for key in LAND_RESIDENCE_KEYS:
                old = float(node[key]); new = old * land_mult; node[key] = new
                log_change(changes, scenario_id, "hydrology", f"nodes/{node_name}/{key}", old, new, f"multiply by {land_mult}")
            for surface_name, surface in iter_surface_items(node):
                path0 = f"nodes/{node_name}/surfaces/{surface_name}"
                for key, pname in [
                    ("surface_coefficient", "surface_coefficient_multiplier"),
                    ("ihacres_p", "ihacres_p_multiplier"),
                    ("percolation_coefficient", "percolation_coefficient_multiplier"),
                ]:
                    if key in surface:
                        old = float(surface[key]); mult = p[pname]; new = old * mult; surface[key] = new
                        log_change(changes, scenario_id, "hydrology", f"{path0}/{key}", old, new, f"multiply by {mult}")
                storage_mult = p["soil_storage_multiplier"]
                for key in ["wilting_point", "field_capacity", "total_porosity"]:
                    if key in surface:
                        old = float(surface[key]); new = old * storage_mult; surface[key] = new
                        log_change(changes, scenario_id, "hydrology", f"{path0}/{key}", old, new, f"multiply by {storage_mult}")
        elif node.get("type_") == "Groundwater":
            rt_mult = p["groundwater_residence_multiplier"]
            old = float(node["residence_time"]); new = old * rt_mult; node["residence_time"] = new
            log_change(changes, scenario_id, "hydrology", f"nodes/{node_name}/residence_time", old, new, f"multiply by {rt_mult}")
            storage_mult = p["groundwater_storage_multiplier"]
            for key in ["capacity", "initial_storage"]:
                old = float(node[key]); new = old * storage_mult; node[key] = new
                log_change(changes, scenario_id, "hydrology", f"nodes/{node_name}/{key}", old, new, f"multiply by {storage_mult}")

    wwtw_value = p["wwtw_nitrate_constant"]
    for node_name in target_wwtws:
        nitrate = cfg["nodes"][node_name]["process_parameters"]["nitrate"]
        old = float(nitrate["constant"]); nitrate["constant"] = wwtw_value
        log_change(changes, scenario_id, "wwtw", f"nodes/{node_name}/process_parameters/nitrate/constant", old, wwtw_value, "globally shared raw nitrate constant")

    # Mandatory pre-processing Soil-N field. The Phase-A process multiplier acts on
    # this field; it never replaces or normalises it.
    soil_n_process_multiplier = p["minfpar_N_process_multiplier"]
    minfpar_by_node_group = {
        node: {
            group: SOIL_N_REFERENCE_MINFPAR * SOIL_N_PREPROCESSING_FACTOR_BY_NODE[node] * soil_n_process_multiplier
            for group in SURFACE_GROUPS
        }
        for node in SOIL_N_PREPROCESSING_FACTOR_BY_NODE
    }
    surface_effective = surface_inventory.copy()
    surface_effective["preprocessing_factor"] = surface_effective["node"].map(SOIL_N_PREPROCESSING_FACTOR_BY_NODE)
    surface_effective["effective_factor"] = surface_effective["preprocessing_factor"] * soil_n_process_multiplier
    audit_area_weighted_effective_multiplier = float(np.average(
        surface_effective["effective_factor"], weights=surface_effective["area"]
    ))
    runtime = {
        "denpar": p["denpar"],
        "minfpar_N_reference": SOIL_N_REFERENCE_MINFPAR,
        "surface_to_group": SURFACE_TO_GROUP,
        "soil_n_node_classification": SOIL_N_NODE_CLASSIFICATION,
        "soil_n_preprocessing_factor_by_node": SOIL_N_PREPROCESSING_FACTOR_BY_NODE,
        "minfpar_N_process_multiplier": soil_n_process_multiplier,
        "audit_area_weighted_effective_multiplier": audit_area_weighted_effective_multiplier,
        "whole_model_normalisation_applied": False,
        "minfpar_N_by_node_group": minfpar_by_node_group,
    }
    if not np.isclose(p["denpar"], BASE_PARAMETERS["denpar"]):
        log_change(
            changes, scenario_id, "soil_nitrate", "runtime/all_GrowingSurface/denpar",
            BASE_PARAMETERS["denpar"], p["denpar"], "Phase-A Soil-N direction/strength",
        )
    for node_name, preprocessing_factor in SOIL_N_PREPROCESSING_FACTOR_BY_NODE.items():
        target = SOIL_N_REFERENCE_MINFPAR * preprocessing_factor * soil_n_process_multiplier
        log_change(
            changes, scenario_id, "soil_nitrate",
            f"runtime/nodes/{node_name}/all_NutrientPool/minfpar/N",
            SOIL_N_REFERENCE_MINFPAR, target,
            "pre-processing mixture-dominance regionalisation multiplied by Phase-A process intensity",
        )
    return cfg, p, runtime, pd.DataFrame(changes)

def audit_scenario_config(scenario_id: str, cfg: dict, parameters: dict, runtime: dict, changes: pd.DataFrame) -> pd.DataFrame:
    checks = []
    add = lambda check, passed, detail: checks.append({"scenario_id": scenario_id, "check": check, "passed": bool(passed), "detail": str(detail)})
    add("corrected_WWTW_routing_and_all_arcs_locked", cfg.get("arcs") == baseline_config.get("arcs"), f"arcs={len(cfg.get('arcs', {}))}")
    add("orchestration_locked", cfg.get("orchestration") == baseline_config.get("orchestration"), "exact equality")
    add("pollutant_lists_locked", all(cfg.get(k) == baseline_config.get(k) for k in ["pollutants", "additive_pollutants", "non_additive_pollutants"]), "exact equality")
    regionalised_rule_id = REGIONALISED_CANDIDATE_BY_INDEX[int(round(parameters['regionalised_rule_index']))]
    regionalised_base = composite_rule_configs[regionalised_rule_id]
    add("regionalised_rule_declared", regionalised_rule_id in composite_rule_configs, regionalised_rule_id)

    for node_name, node in cfg["nodes"].items():
        if node.get("type_") == "Land":
            base_node = regionalised_base["nodes"][node_name]
            for key in LAND_RESIDENCE_KEYS:
                expected = float(base_node[key]) * parameters["land_residence_multiplier"]
                add("regionalised_land_residence_multiplier", np.isclose(float(node[key]), expected), f"{regionalised_rule_id}/{node_name}/{key}")
            add("positive_land_residence", all(float(node[k]) > 0 for k in LAND_RESIDENCE_KEYS), node_name)
            base_surfaces = dict(iter_surface_items(base_node))
            for surface_name, surface in iter_surface_items(node):
                base_surface = base_surfaces[surface_name]
                for key, pname in [("surface_coefficient", "surface_coefficient_multiplier"), ("ihacres_p", "ihacres_p_multiplier"), ("percolation_coefficient", "percolation_coefficient_multiplier")]:
                    if key in surface:
                        expected = float(base_surface[key]) * parameters[pname]
                        add("regionalised_surface_multiplier", np.isclose(float(surface[key]), expected), f"{regionalised_rule_id}/{node_name}/{surface_name}/{key}")
                for key in ["wilting_point", "field_capacity", "total_porosity"]:
                    if key in surface:
                        expected = float(base_surface[key]) * parameters["soil_storage_multiplier"]
                        add("regionalised_combined_soil_storage", np.isclose(float(surface[key]), expected), f"{regionalised_rule_id}/{node_name}/{surface_name}/{key}")
                if all(k in surface for k in ["wilting_point", "field_capacity", "total_porosity"]):
                    ok = 0 < float(surface["wilting_point"]) < float(surface["field_capacity"]) < float(surface["total_porosity"]) <= 1
                    add("soil_storage_order", ok, f"{node_name}/{surface_name}")
                if "surface_coefficient" in surface:
                    add("surface_coefficient_bounds", 0 <= float(surface["surface_coefficient"]) <= 1, f"{node_name}/{surface_name}")
                if "ihacres_p" in surface:
                    add("ihacres_p_bounds", 0 <= float(surface["ihacres_p"]) <= 15, f"{node_name}/{surface_name}")
                if "percolation_coefficient" in surface:
                    add("percolation_bounds", 0 <= float(surface["percolation_coefficient"]) <= 1, f"{node_name}/{surface_name}")
        elif node.get("type_") == "Groundwater":
            base_node = regionalised_base["nodes"][node_name]
            add("regionalised_groundwater_residence", np.isclose(float(node["residence_time"]), float(base_node["residence_time"]) * parameters["groundwater_residence_multiplier"]), f"{regionalised_rule_id}/{node_name}")
            storage_ratio_ok = all(np.isclose(float(node[key]), float(base_node[key]) * parameters["groundwater_storage_multiplier"]) for key in ["capacity", "initial_storage"])
            add("regionalised_combined_groundwater_storage", storage_ratio_ok, f"{regionalised_rule_id}/{node_name}")
            ok = float(node["capacity"]) > 0 and 0 <= float(node["initial_storage"]) <= float(node["capacity"]) and float(node["residence_time"]) > 0
            add("groundwater_constraints", ok, node_name)

    for node_name in all_wwtws:
        current = cfg["nodes"][node_name]
        base = regionalised_base["nodes"][node_name]
        current_n = current["process_parameters"]["nitrate"]
        base_n = base["process_parameters"]["nitrate"]
        constant_ok = np.isclose(current_n["constant"], parameters["wwtw_nitrate_constant"]) if node_name in target_wwtws else np.isclose(current_n["constant"], base_n["constant"])
        other_ok = (
            np.isclose(current_n["exponent"], base_n["exponent"])
            and np.isclose(current["treatment_throughput_capacity"], base["treatment_throughput_capacity"])
            and current.get("liquor_multiplier") == base.get("liquor_multiplier")
        )
        add("wwtw_global_shared_constant_and_structure_lock", constant_ok and other_ok, node_name)

    expected_process = float(parameters["minfpar_N_process_multiplier"])
    preprocessing_persists = all(
        np.isclose(
            float(runtime["minfpar_N_by_node_group"][node][group]),
            SOIL_N_REFERENCE_MINFPAR * SOIL_N_PREPROCESSING_FACTOR_BY_NODE[node] * expected_process,
            rtol=1e-12, atol=1e-12,
        )
        for node in SOIL_N_PREPROCESSING_FACTOR_BY_NODE for group in SURFACE_GROUPS
    )
    add("soil_N_preprocessing_regionalisation_persists", preprocessing_persists, f"process_multiplier={expected_process}")
    add("soil_N_no_whole_model_normalisation", runtime.get("whole_model_normalisation_applied") is False, "explicit contract")
    if np.isclose(expected_process, 1.0):
        n0_exact = all(
            np.isclose(
                float(runtime["minfpar_N_by_node_group"][node][group]),
                SOIL_N_REFERENCE_MINFPAR * SOIL_N_PREPROCESSING_FACTOR_BY_NODE[node],
                rtol=1e-12, atol=1e-12,
            )
            for node in SOIL_N_PREPROCESSING_FACTOR_BY_NODE for group in SURFACE_GROUPS
        )
        add("soil_N_N0_equals_preprocessing_not_corrected_baseline", n0_exact, "N0 retains F_preprocessing")
    if not changes.empty:
        allowed_groups = {"hydrology", "wwtw", "soil_nitrate"}
        add("change_groups_registered", set(changes["group"]).issubset(allowed_groups), sorted(set(changes["group"])))
    return pd.DataFrame(checks)

def materialise_scenarios(design: pd.DataFrame, phase: str, warmup_start: str, warmup_end: str):
    expected_warmup_days = len(pd.date_range(warmup_start, warmup_end, freq="D"))
    allowed_windows = {
        (COARSE_WARMUP_START, COARSE_WARMUP_END, len(expected_coarse_warmup)),
        (FINE_WARMUP_START, FINE_WARMUP_END, len(expected_fine_warmup)),
    }
    if (warmup_start, warmup_end, expected_warmup_days) not in allowed_windows:
        raise RuntimeError(f"Undeclared warm-up window for {phase}: {warmup_start} to {warmup_end}")
    design_rows, change_frames, audit_frames, spec_rows = [], [], [], []
    for row in design.to_dict("records"):
        sid = row["scenario_id"]
        params = json.loads(row["parameters_json"])
        cfg, full_params, runtime, changes = build_scenario_config(sid, params)
        audits = audit_scenario_config(sid, cfg, full_params, runtime, changes)
        if not audits["passed"].all():
            display(audits[~audits["passed"]])
            raise RuntimeError(f"Scenario audit failed: {sid}")
        config_path = CONFIG_DIR / f"{sid}.yml"
        config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
        parameter_fingerprint = hashlib.sha256(json.dumps(full_params, sort_keys=True).encode()).hexdigest()
        generated_config_sha256=sha256(config_path)
        runtime_overrides_sha256=hashlib.sha256(json.dumps(runtime,sort_keys=True,separators=(",",":")).encode()).hexdigest()
        source_contract_sha256=hashlib.sha256(json.dumps(SOURCE_CONTRACT,sort_keys=True,separators=(",",":")).encode()).hexdigest()
        preprocessing_generator_manifest_sha256=sha256(REGIONAL_GENERATOR_MANIFEST)
        run_contract_payload = {
            "framework_implementation_version":FRAMEWORK_IMPLEMENTATION_VERSION,
            "parameters": full_params, "phase": phase,
            "warmup_start": warmup_start, "warmup_end": warmup_end,
            "output_start": OUTPUT_START, "output_end": OUTPUT_END,
            "routing_baseline_sha256": SOURCE_CONTRACT[str(ROUTING_BASELINE_CONFIG)],
            "generated_config_sha256":generated_config_sha256, "runtime_overrides_sha256":runtime_overrides_sha256,
            "source_contract_sha256":source_contract_sha256, "preprocessing_generator_manifest_sha256":preprocessing_generator_manifest_sha256,
            "observation_matrix_bundle_sha256": OBS_MATRIX_BUNDLE_SHA256,
            "soil_N_landcover_rule_contract_sha256": SOIL_N_RULE_CONTRACT_SHA256,
            "global_WWTW_scope_sha256": WWTW_SCOPE_SHA256,
        }
        run_fingerprint = hashlib.sha256(json.dumps(run_contract_payload, sort_keys=True).encode()).hexdigest()
        spec = {
            "scenario_id": sid,
            "phase": phase,
            "MODEL_ROOT": str(MODEL_ROOT),
            "ORIGINAL_WSI_ROOT": str(ORIGINAL_WSI_ROOT),
            "config_path": str(config_path),
            "scenario_output_dir": str(RUN_DIR / sid),
            "record_arcs": record_arcs,
            "runtime_overrides": runtime,
            "parameters": full_params,
            "parameter_fingerprint": parameter_fingerprint,
            "run_fingerprint": run_fingerprint,
            "framework_implementation_version":FRAMEWORK_IMPLEMENTATION_VERSION,
            "generated_config_sha256":generated_config_sha256, "runtime_overrides_sha256":runtime_overrides_sha256,
            "source_contract_sha256":source_contract_sha256, "preprocessing_generator_manifest_sha256":preprocessing_generator_manifest_sha256,
            "source_contract": SOURCE_CONTRACT,
            "observation_matrix_bundle_sha256": OBS_MATRIX_BUNDLE_SHA256,
            "soil_N_landcover_rule_contract_sha256": SOIL_N_RULE_CONTRACT_SHA256,
            "global_WWTW_scope_sha256": WWTW_SCOPE_SHA256,
            "WARMUP_START": warmup_start, "WARMUP_END": warmup_end,
            "EXPECTED_WARMUP_DAYS": expected_warmup_days,
            "OUTPUT_START": OUTPUT_START, "OUTPUT_END": OUTPUT_END,
            "VALIDATION_CHUNK_DAYS": VALIDATION_CHUNK_DAYS,
            "FORCE_RERUN": bool(FORCE_RERUN),
        }
        spec_path = SPEC_DIR / f"{sid}.json"
        spec_path.write_text(json.dumps(spec, indent=2), encoding="utf-8")
        design_rows.append({**row, **full_params, "phase": phase, "warmup_start": warmup_start, "warmup_end": warmup_end, "warmup_days": expected_warmup_days, "config_path": str(config_path), "parameter_fingerprint": parameter_fingerprint, "run_fingerprint": run_fingerprint})
        if not changes.empty: change_frames.append(changes)
        audit_frames.append(audits)
        spec_rows.append({"scenario_id": sid, "phase": phase, "warmup_start": warmup_start, "warmup_end": warmup_end, "warmup_days": expected_warmup_days, "run_fingerprint": run_fingerprint, "spec_path": str(spec_path), "output_csv": str(RUN_DIR / sid / "model_outputs_2014_2019.csv")})
    materialised = pd.DataFrame(design_rows)
    changes_all = pd.concat(change_frames, ignore_index=True) if change_frames else pd.DataFrame(columns=["scenario_id", "group", "path", "old_value", "new_value", "rule"])
    audits_all = pd.concat(audit_frames, ignore_index=True)
    specs = pd.DataFrame(spec_rows)
    materialised.to_csv(TABLE_DIR / f"{phase}_scenario_design.csv", index=False)
    changes_all.to_csv(TABLE_DIR / f"{phase}_complete_change_log.csv", index=False)
    audits_all.to_csv(TABLE_DIR / f"{phase}_audit.csv", index=False)
    specs.to_csv(TABLE_DIR / f"{phase}_run_specs.csv", index=False)
    assert_sources_unchanged()
    return materialised, changes_all, audits_all, specs


## 5. Phase A: separate calibration-domain experiments

**Dissertation correspondence:** Section 2.5.

**What this step does:** Creates complete Hydrology vectors and independent WWTW and Soil-N direction/strength tests

**Checks / outputs:** Phase-A scenario library plus complete-H and Soil-N inheritance contracts


In [ ]:
# Framework correspondence: Section 2.5.
# What this cell does (Phase A scenario design): Creates separate Hydrology, WWTW and Soil-N direction-and-intensity experiments.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

def direction_row(
    scenario_id,
    direction_id,
    module,
    hypothesis,
    endpoint_updates=None,
    screening_level=0.0,
    endpoint_strength=1.0,
    regionalised_anchor="R00",
):
    """Materialise one tested level and retain its unscaled direction endpoint."""
    endpoint_updates = endpoint_updates or {}
    endpoint = dict(BASE_PARAMETERS)
    endpoint.update({key: float(value) for key, value in endpoint_updates.items()})
    tested = dict(BASE_PARAMETERS)

    if module == "hydrology":
        fraction = float(screening_level) / float(endpoint_strength)
    elif module in {"wwtw", "soil_nitrate"}:
        fraction = float(screening_level)
    else:
        fraction = 0.0

    for parameter, rule in PARAMETER_RULES.items():
        if rule["group"] != module or parameter == "regionalised_rule_index":
            continue
        tested[parameter] = BASE_PARAMETERS[parameter] + fraction * (
            endpoint[parameter] - BASE_PARAMETERS[parameter]
        )

    if module == "hydrology":
        tested_key = (regionalised_anchor, float(screening_level))
        endpoint_key = (regionalised_anchor, float(endpoint_strength))
        if tested_key not in REGIONALISED_WEIGHT_INDEX or endpoint_key not in REGIONALISED_WEIGHT_INDEX:
            raise RuntimeError(f"Missing regionalised candidate: tested={tested_key}, endpoint={endpoint_key}")
        tested["regionalised_rule_index"] = float(REGIONALISED_WEIGHT_INDEX[tested_key])
        endpoint["regionalised_rule_index"] = float(REGIONALISED_WEIGHT_INDEX[endpoint_key])

    for name, value in tested.items():
        rule = PARAMETER_RULES[name]
        if not (float(rule["lower"]) <= float(value) <= float(rule["upper"])):
            raise RuntimeError(f"Phase-A tested value outside bounds: {scenario_id} {name}={value}")

    return {
        "scenario_id": scenario_id,
        "direction_id": direction_id,
        "module": module,
        "physical_hypothesis": hypothesis,
        "regionalised_anchor": regionalised_anchor,
        "screening_level": float(screening_level),
        "endpoint_strength": float(endpoint_strength),
        "parameters_json": json.dumps(tested, sort_keys=True),
        "endpoint_parameters_json": json.dumps(endpoint, sort_keys=True),
    }


phase_a_rows = [direction_row(
    "A00_BASELINE", "A00_BASELINE", "baseline",
    "confirmed routing plus mandatory pre-processing Soil-N regionalisation and no Phase-A process perturbation",
    screening_level=0.0,
    endpoint_strength=0.0,
)]

# Complete hydrology endpoint vectors. These values are applied together.
FAST_EXPORT_H = {
    "surface_coefficient_multiplier": 1.2,
    "ihacres_p_multiplier": 1.2,
    "land_residence_multiplier": 0.8,
    "percolation_coefficient_multiplier": 0.8,
    "soil_storage_multiplier": 0.9,
    "groundwater_residence_multiplier": 0.8,
    "groundwater_storage_multiplier": 1.2,
}
SLOW_RETENTIVE_H = {
    "surface_coefficient_multiplier": 0.8,
    "ihacres_p_multiplier": 0.8,
    "land_residence_multiplier": 1.2,
    "percolation_coefficient_multiplier": 1.2,
    "soil_storage_multiplier": 1.1,
    "groundwater_residence_multiplier": 1.2,
    "groundwater_storage_multiplier": 0.8,
}
HYDROLOGY_PROCESS_BUNDLES = {
    "FAST_EXPORT": FAST_EXPORT_H,
    "SLOW_RETENTIVE": SLOW_RETENTIVE_H,
}

for rule_number in range(1, 8):
    rule_id = f"R{rule_number:02d}"
    for bundle_name, updates in HYDROLOGY_PROCESS_BUNDLES.items():
        direction_id = f"H_{rule_id}_{bundle_name}"
        for level in PHASE_A_H_SCREEN_LEVELS:
            phase_a_rows.append(direction_row(
                f"A_{direction_id}_H{int(round(level * 100)):03d}",
                direction_id,
                "hydrology",
                f"{rule_id} complete {bundle_name.lower()} direction tested at H{level:g}",
                updates,
                screening_level=level,
                endpoint_strength=HYDROLOGY_ENDPOINT_STRENGTH,
                regionalised_anchor=rule_id,
            ))

# WWTW remains one globally shared raw nitrate constant.
for direction_id, target in [("W_C18", 18.0), ("W_C22", 22.0)]:
    for level in PHASE_A_W_SCREEN_LEVELS:
        phase_a_rows.append(direction_row(
            f"A_{direction_id}_W{int(round(level * 100)):03d}",
            direction_id,
            "wwtw",
            f"global WWTW direction {direction_id} tested at W{level:g}",
            {"wwtw_nitrate_constant": target},
            screening_level=level,
        ))

# Soil-N class multipliers mi are fixed evidence inputs in pre-processing. Phase A
# varies only process direction and strength on top of that regionalised field.
nitrate_directions = [
    ("N_MORE_AVAILABLE", "higher mineralisation plus lower denitrification", {"minfpar_N_process_multiplier": 2.0, "denpar": 0.012}),
    ("N_LESS_AVAILABLE", "lower mineralisation plus higher denitrification", {"minfpar_N_process_multiplier": 0.5, "denpar": 0.018}),
    ("N_HIGH_TURNOVER", "higher mineralisation and higher denitrification", {"minfpar_N_process_multiplier": 2.0, "denpar": 0.018}),
    ("N_LOW_TURNOVER", "lower mineralisation and lower denitrification", {"minfpar_N_process_multiplier": 0.5, "denpar": 0.012}),
]
for direction_id, hypothesis, updates in nitrate_directions:
    for level in PHASE_A_N_SCREEN_LEVELS:
        phase_a_rows.append(direction_row(
            f"A_{direction_id}_N{int(round(level * 100)):03d}",
            direction_id,
            "soil_nitrate",
            f"{hypothesis}, tested at N{level:g}",
            updates,
            screening_level=level,
        ))

phase_a_grid = pd.DataFrame(phase_a_rows)
if phase_a_grid["scenario_id"].duplicated().any():
    raise RuntimeError("Duplicated Phase-A scenario IDs")

# Prove that every nonbaseline H row changes the spatial field and all seven H modifiers.
hydrology_parameters = [
    name for name, rule in PARAMETER_RULES.items()
    if rule["group"] == "hydrology" and name != "regionalised_rule_index"
]
hydrology_contract_rows = []
for row in phase_a_grid.loc[phase_a_grid["module"].eq("hydrology")].itertuples(index=False):
    params = json.loads(row.parameters_json)
    changed = [name for name in hydrology_parameters if not np.isclose(params[name], BASE_PARAMETERS[name])]
    passed = (
        row.regionalised_anchor != "R00"
        and row.screening_level in PHASE_A_H_SCREEN_LEVELS
        and set(changed) == set(hydrology_parameters)
    )
    hydrology_contract_rows.append({
        "scenario_id": row.scenario_id,
        "direction_id": row.direction_id,
        "regionalised_anchor": row.regionalised_anchor,
        "screening_level": row.screening_level,
        "changed_continuous_H_parameters": ";".join(changed),
        "complete_H_contract_passed": passed,
    })
hydrology_contract = pd.DataFrame(hydrology_contract_rows)
if not hydrology_contract["complete_H_contract_passed"].all():
    raise RuntimeError("At least one Phase-A H candidate is not a complete hydrology vector")

soil_n_contract_rows = []
for row in phase_a_grid.loc[phase_a_grid["module"].eq("soil_nitrate")].itertuples(index=False):
    params = json.loads(row.parameters_json)
    soil_n_contract_rows.append({
        "scenario_id": row.scenario_id,
        "direction_id": row.direction_id,
        "screening_level": row.screening_level,
        "preprocessing_factor_source": "SOIL_N_PREPROCESSING_FACTOR_BY_NODE",
        "preprocessing_factor_is_optimised": False,
        "minfpar_N_process_multiplier": params["minfpar_N_process_multiplier"],
        "denpar": params["denpar"],
        "no_class_contrast_optimisation": not any("_contrast" in name for name in params),
        "preprocessing_inheritance_contract_passed": (
            row.screening_level in PHASE_A_N_SCREEN_LEVELS
            and not any("_contrast" in name for name in params)
        ),
    })
soil_n_contract = pd.DataFrame(soil_n_contract_rows)
if not soil_n_contract["preprocessing_inheritance_contract_passed"].all():
    raise RuntimeError("At least one Phase-A Soil-N candidate hides or re-optimises the pre-processing regionalisation")

phase_a_grid.to_csv(TABLE_DIR / "02_phase_A_coupled_direction_library.csv", index=False)
hydrology_contract.to_csv(TABLE_DIR / "02_phase_A_complete_H_contract_audit.csv", index=False)
soil_n_contract.to_csv(TABLE_DIR / "02_phase_A_soil_N_preprocessing_inheritance_audit.csv", index=False)
phase_a_design, phase_a_changes, phase_a_audits, phase_a_specs = materialise_scenarios(
    phase_a_grid, "phase_A_complete_H_5y", COARSE_WARMUP_START, COARSE_WARMUP_END
)
print("Phase-A scenarios:", len(phase_a_design))
display(phase_a_grid[[
    "scenario_id", "module", "physical_hypothesis",
    "direction_id", "regionalised_anchor", "screening_level", "endpoint_strength"
]])
display(hydrology_contract)


## 6. Common simulation engine

**Dissertation correspondence:** Section 2.2, applied consistently in Sections 2.5 and 2.6.

**What this step does:** Runs or verifies cached Phase-A and Phase-B scenarios under one WSIMOD, warm-up, output and source-hash contract

**Checks / outputs:** Per-scenario runtime parameter, output-completeness and timing audits


In [ ]:
# Framework correspondence: Section 2.2.
# What this cell does (Common simulation engine): Runs every declared Phase-A or Phase-B scenario under the same warm-up, output and audit contract.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

WORKER_CODE = r'''
from __future__ import annotations
import hashlib, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

spec = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
original_wsi = Path(spec["ORIGINAL_WSI_ROOT"]).resolve()
sys.path.insert(0, str(original_wsi))
import wsimod
from wsimod.orchestration.model import Model

imported = Path(wsimod.__file__).resolve()
if original_wsi not in imported.parents:
    raise RuntimeError(f"Wrong WSIMOD import: {imported}; expected under {original_wsi}")

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

for path, expected in spec["source_contract"].items():
    if sha256(path) != expected:
        raise RuntimeError(f"Immutable source hash mismatch before run: {path}")

out = Path(spec["scenario_output_dir"]); out.mkdir(parents=True, exist_ok=True)
output_csv = out / "model_outputs_2014_2019.csv"
summary_json = out / "run_summary.json"
progress_csv = out / "progress.csv"
runtime_audit_csv = out / "runtime_parameter_audit.csv"
runtime_landcover_csv = out / "runtime_landcover_effective_multiplier.csv"
if summary_json.exists() and output_csv.exists() and not spec.get("FORCE_RERUN", False):
    old = json.loads(summary_json.read_text(encoding="utf-8"))
    if old.get("status") == "complete" and old.get("run_fingerprint") == spec["run_fingerprint"]:
        print(json.dumps({"scenario_id":spec["scenario_id"],"status":"skipped_valid_existing"})); raise SystemExit(0)
for path in [output_csv, summary_json, progress_csv, runtime_audit_csv, runtime_landcover_csv]:
    if path.exists(): path.unlink()

model = Model()
model.load(spec["MODEL_ROOT"], config_name=spec["config_path"])
runtime = spec["runtime_overrides"]
rows=[]; minfpar_dict_ids=[]
for land_name, land in model.nodes_type.get("Land", {}).items():
    for surface in land.surfaces:
        surface_name = str(getattr(surface, "surface", surface.__class__.__name__))
        surface_group = runtime["surface_to_group"].get(surface_name)
        if hasattr(surface, "denpar"):
            old=float(surface.denpar); surface.apply_overrides({"denpar":float(runtime["denpar"])})
            rows.append({"node":land_name,"surface":surface_name,"surface_group":surface_group,"parameter":"denpar","old_value":old,"new_value":float(surface.denpar),"expected_value":float(runtime["denpar"])})
        if hasattr(surface, "nutrient_pool"):
            if land_name not in runtime["minfpar_N_by_node_group"] or surface_group not in runtime["minfpar_N_by_node_group"][land_name]:
                raise RuntimeError(f"Unmapped thresholded nutrient-pool surface: {land_name}/{surface_name}")
            old=float(surface.nutrient_pool.minfpar["N"])
            old_P=float(surface.nutrient_pool.minfpar["P"]); target_N=float(runtime["minfpar_N_by_node_group"][land_name][surface_group])
            private=dict(surface.nutrient_pool.minfpar); private["N"]=target_N; surface.nutrient_pool.minfpar=private
            minfpar_dict_ids.append(id(surface.nutrient_pool.minfpar))
            meta=runtime["soil_n_node_classification"][land_name]
            rows.append({"node":land_name,"surface":surface_name,"surface_group":surface_group,"surface_area":float(surface.area),"parameter":"minfpar_N","old_value":old,"new_value":float(surface.nutrient_pool.minfpar["N"]),"expected_value":target_N,"effective_multiplier":target_N/float(runtime["minfpar_N_reference"]),"dominant_group":meta["dominant_group"],"dominant_share":meta["dominant_share"],"threshold_stage":meta["threshold_stage"],"dominant_blend_alpha":meta["dominant_blend_alpha"],"old_P":old_P,"new_P":float(surface.nutrient_pool.minfpar["P"])})
audit=pd.DataFrame(rows); audit.to_csv(runtime_audit_csv,index=False)
den=audit[audit.parameter.eq("denpar")]; mineral=audit[audit.parameter.eq("minfpar_N")]
global_effective=np.average(mineral["effective_multiplier"],weights=mineral["surface_area"])
runtime_ok=(len(den)==827 and len(mineral)==827 and len(set(minfpar_dict_ids))==827 and not mineral.surface_group.isna().any() and np.isclose(global_effective,float(runtime["audit_area_weighted_effective_multiplier"]),rtol=1e-12,atol=1e-12) and np.isclose(den.new_value,den.expected_value,rtol=1e-12,atol=0).all() and np.isclose(mineral.new_value,mineral.expected_value,rtol=1e-12,atol=0).all() and np.isclose(mineral.old_P,mineral.new_P,rtol=1e-12,atol=0).all())
if not runtime_ok:
    raise RuntimeError("Runtime parameter audit failed")
catchment_effective=mineral.groupby(["node","dominant_group","dominant_share","threshold_stage","dominant_blend_alpha"]).apply(lambda frame: np.average(frame["effective_multiplier"],weights=frame["surface_area"])).rename("area_weighted_effective_minfpar_N_multiplier").reset_index()
catchment_effective["preprocessing_regionalisation_factor"]=catchment_effective["node"].map(runtime["soil_n_preprocessing_factor_by_node"])
catchment_effective["phase_A_process_multiplier"]=float(runtime["minfpar_N_process_multiplier"])
catchment_effective.to_csv(runtime_landcover_csv,index=False)

warmup=pd.date_range(spec["WARMUP_START"],spec["WARMUP_END"],freq="D")
output_dates=pd.date_range(spec["OUTPUT_START"],spec["OUTPUT_END"],freq="D")
if len(warmup)!=int(spec["EXPECTED_WARMUP_DAYS"]): raise RuntimeError(f"Warm-up contract mismatch: {len(warmup)} vs {spec['EXPECTED_WARMUP_DAYS']} days")
started=time.perf_counter(); progress=[]
model.run(dates=warmup,record_arcs=[],record_all=False,verbose=False)
progress.append({"stage":"warmup","start":str(warmup.min().date()),"end":str(warmup.max().date()),"days":len(warmup),"elapsed_minutes":(time.perf_counter()-started)/60})
pd.DataFrame(progress).to_csv(progress_csv,index=False)
chunk_days=int(spec["VALIDATION_CHUNK_DAYS"])
for i in range(0,len(output_dates),chunk_days):
    chunk=output_dates[i:i+chunk_days]
    flows,*_=model.run(dates=chunk,record_arcs=spec["record_arcs"],record_all=False,verbose=False)
    flows=flows if isinstance(flows,pd.DataFrame) else pd.DataFrame(flows)
    flows.to_csv(output_csv,mode="a",header=not output_csv.exists(),index=False)
    progress.append({"stage":f"output_chunk_{i//chunk_days+1}","start":str(chunk.min().date()),"end":str(chunk.max().date()),"days":len(chunk),"rows":len(flows),"elapsed_minutes":(time.perf_counter()-started)/60})
    pd.DataFrame(progress).to_csv(progress_csv,index=False)
check=pd.read_csv(output_csv,usecols=["arc","time"])
expected=len(output_dates)*len(spec["record_arcs"])
duplicates=int(check.duplicated(["arc","time"]).sum())
if len(check)!=expected or check.arc.nunique()!=len(spec["record_arcs"]) or check.time.nunique()!=len(output_dates) or duplicates:
    raise RuntimeError(f"Output gate failed rows={len(check)}/{expected}, arcs={check.arc.nunique()}, dates={check.time.nunique()}, duplicates={duplicates}")
for path, expected_hash in spec["source_contract"].items():
    if sha256(path) != expected_hash: raise RuntimeError(f"Immutable source hash mismatch after run: {path}")
summary={"scenario_id":spec["scenario_id"],"phase":spec["phase"],"status":"complete","framework_implementation_version":spec["framework_implementation_version"],"generated_config_sha256":spec["generated_config_sha256"],"runtime_overrides_sha256":spec["runtime_overrides_sha256"],"source_contract_sha256":spec["source_contract_sha256"],"preprocessing_generator_manifest_sha256":spec["preprocessing_generator_manifest_sha256"],"elapsed_minutes":(time.perf_counter()-started)/60,"warmup_start":spec["WARMUP_START"],"warmup_end":spec["WARMUP_END"],"warmup_days":len(warmup),"output_days":len(output_dates),"output_rows":len(check),"parameter_fingerprint":spec["parameter_fingerprint"],"run_fingerprint":spec["run_fingerprint"],"observation_matrix_bundle_sha256":spec["observation_matrix_bundle_sha256"],"soil_N_landcover_rule_contract_sha256":spec["soil_N_landcover_rule_contract_sha256"],"wsimod_import":str(imported),"config_path":spec["config_path"],"runtime_audit_csv":str(runtime_audit_csv),"runtime_landcover_effective_multiplier_csv":str(runtime_landcover_csv)}
summary_json.write_text(json.dumps(summary,indent=2),encoding="utf-8"); print(json.dumps(summary))
'''

def run_specs(specs: pd.DataFrame, label: str) -> pd.DataFrame:
    assert_sources_unchanged()
    def run_one(row):
        sid = row.scenario_id
        completed = subprocess.run(
            [str(WSIMOD_PYTHON), "-c", WORKER_CODE, row.spec_path],
            text=True, capture_output=True,
        )
        out = RUN_DIR / sid; out.mkdir(parents=True, exist_ok=True)
        (out / "stdout.log").write_text(completed.stdout or "", encoding="utf-8")
        (out / "stderr.log").write_text(completed.stderr or "", encoding="utf-8")
        return {"scenario_id": sid, "returncode": completed.returncode, "stderr_tail": (completed.stderr or "")[-1000:]}
    results=[]
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_WORKERS) as executor:
        futures={executor.submit(run_one,row):row.scenario_id for row in specs.itertuples(index=False)}
        for number,future in enumerate(as_completed(futures),start=1):
            result=future.result(); results.append(result)
            print(f"[{number}/{len(futures)}] {result['scenario_id']} returncode={result['returncode']}")
    result=pd.DataFrame(results).sort_values("scenario_id")
    result.to_csv(TABLE_DIR / f"{label}_subprocess_results.csv",index=False)
    if result["returncode"].ne(0).any():
        display(result[result["returncode"].ne(0)])
        raise RuntimeError(f"{label} contains failed scenarios; inspect stderr.log")
    assert_sources_unchanged()
    return result


In [ ]:
# Framework correspondence: Section 2.5.
# What this cell does (Phase A execution): Executes or validates cached separate-domain Phase-A scenarios.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

if RUN_PHASE_A:
    phase_a_run_results = run_specs(phase_a_specs, "phase_A_5y")
    display(phase_a_run_results)
else:
    print("Phase A not launched. Review tables/01_parameter_registry.csv and tables/phase_A_complete_change_log.csv, then set RUN_PHASE_A=True.")


## 7. Evaluation protocol

**Dissertation correspondence:** Section 2.2.

**What this step does:** Calculates paired flow and nitrate-concentration KGE for calibration and validation periods; nitrate load is retained as supplementary evidence

**Checks / outputs:** Metric tables use the confirmed station mapping and explicitly remove invalid pairs


In [ ]:
# Framework correspondence: Section 2.2.
# What this cell does (Evaluation protocol): Calculates paired flow and nitrate KGE consistently for calibration and validation periods.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

PERIODS = {
    "calibration": (pd.Timestamp(CALIBRATION_START), pd.Timestamp(CALIBRATION_END)),
    "validation": (pd.Timestamp(VALIDATION_START), pd.Timestamp(VALIDATION_END)),
    "full_2014_2019": (pd.Timestamp(OUTPUT_START), pd.Timestamp(OUTPUT_END)),
}

def kge_components(sim, obs):
    x=pd.to_numeric(sim,errors="coerce"); y=pd.to_numeric(obs,errors="coerce")
    keep=x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y); x=x[keep].astype(float); y=y[keep].astype(float)
    if len(x)<2 or np.isclose(y.std(ddof=0),0) or np.isclose(y.mean(),0):
        return {"n":len(x),"KGE":np.nan,"r":np.nan,"alpha":np.nan,"beta":np.nan,"PBIAS":np.nan,"mean_sim":np.nan,"mean_obs":np.nan}
    r=float(np.corrcoef(x,y)[0,1]); alpha=float(x.std(ddof=0)/y.std(ddof=0)); beta=float(x.mean()/y.mean())
    return {"n":len(x),"KGE":1-math.sqrt((r-1)**2+(alpha-1)**2+(beta-1)**2),"r":r,"alpha":alpha,"beta":beta,"PBIAS":100*float((x-y).sum()/y.sum()),"mean_sim":float(x.mean()),"mean_obs":float(y.mean())}

def load_model_output(path: Path):
    if not path.exists(): return None
    chunks=[]
    for chunk in pd.read_csv(path,chunksize=200_000):
        chunk=chunk[chunk["arc"].isin(record_arcs)].copy()
        if not chunk.empty:
            chunk["date"]=pd.to_datetime(chunk["time"]).dt.normalize(); chunks.append(chunk)
    return pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame()

def evaluate_scenario(scenario_id: str, output_csv: Path) -> pd.DataFrame:
    output=load_model_output(output_csv)
    if output is None: return pd.DataFrame()
    rows=[]
    for mapping_mode,matrix in observation_matrices.items():
        for assignment in matrix.itertuples(index=False):
            observed=pd.read_csv(assignment.senior_file,parse_dates=["date"])[["date","observation"]]
            observed["date"]=pd.to_datetime(observed["date"]).dt.normalize()
            sim=output[output["arc"].eq(assignment.outfall_arc)].copy()
            if assignment.dataset=="flow": sim["simulation"]=pd.to_numeric(sim["flow"],errors="coerce")/86400.0
            else:
                flow=pd.to_numeric(sim["flow"],errors="coerce"); load=pd.to_numeric(sim["nitrate"],errors="coerce")
                sim["simulation"]=np.where(flow>0,load/flow*1000.0,np.nan)
            paired=sim[["date","simulation"]].merge(observed,on="date",how="inner")
            for period,(start,end) in PERIODS.items():
                subset=paired[(paired["date"]>=start)&(paired["date"]<=end)]
                rows.append({"scenario_id":scenario_id,"mapping_mode":mapping_mode,"period":period,"dataset":assignment.dataset,"catchment_id":assignment.catchment_id,"outfall_arc":assignment.outfall_arc,**kge_components(subset["simulation"],subset["observation"])})
    return pd.DataFrame(rows)

def evaluate_specs(specs: pd.DataFrame, label: str):
    frames=[]; missing=[]
    for row in specs.itertuples(index=False):
        spec=json.loads(Path(row.spec_path).read_text(encoding="utf-8"))
        summary_path=Path(row.output_csv).parent/"run_summary.json"
        if not summary_path.exists(): missing.append(row.scenario_id); continue
        completed=json.loads(summary_path.read_text(encoding="utf-8"))
        if not (completed.get("status")=="complete" and completed.get("run_fingerprint")==spec["run_fingerprint"] and completed.get("framework_implementation_version")==FRAMEWORK_IMPLEMENTATION_VERSION):
            missing.append(row.scenario_id); continue
        frame=evaluate_scenario(row.scenario_id,Path(row.output_csv))
        if frame.empty: missing.append(row.scenario_id)
        else: frames.append(frame)
    if not frames:
        print(f"No completed outputs for {label}"); return pd.DataFrame(),pd.DataFrame(),missing
    metrics=pd.concat(frames,ignore_index=True)
    supplementary_summary=metrics.groupby(["scenario_id","mapping_mode","period","dataset"],as_index=False).agg(n_WFD=("catchment_id","nunique"),mean_KGE=("KGE","mean"),worst_KGE=("KGE","min"),median_KGE=("KGE","median"),mean_PBIAS=("PBIAS","mean"),mean_r=("r","mean"),mean_alpha=("alpha","mean"),mean_beta=("beta","mean"))
    primary=metrics[metrics["catchment_id"].astype(str).isin(PRIMARY_PAIRED_WFDS)].copy()
    calibration_finite=primary[primary["mapping_mode"].eq(SELECTION_MAPPING_MODE)&primary["period"].eq("calibration")&np.isfinite(pd.to_numeric(primary["KGE"],errors="coerce"))]
    selected_counts=calibration_finite.groupby(["scenario_id","dataset"])["catchment_id"].nunique()
    if len(selected_counts)!=len(specs)*2 or not selected_counts.eq(9).all(): raise RuntimeError(f"{label} calibration gate requires 9 finite flow and nitrate KGEs per scenario")
    summary=primary.groupby(["scenario_id","mapping_mode","period","dataset"],as_index=False).agg(n_WFD=("catchment_id","nunique"),mean_KGE=("KGE","mean"),worst_KGE=("KGE","min"),median_KGE=("KGE","median"),mean_PBIAS=("PBIAS","mean"),mean_r=("r","mean"),mean_alpha=("alpha","mean"),mean_beta=("beta","mean"))
    mapping_delta=metrics.pivot_table(index=["scenario_id","period","dataset","catchment_id"],columns="mapping_mode",values="KGE",aggfunc="first").reset_index()
    if {"robin_original","official_station_alternative"}.issubset(mapping_delta.columns):
        mapping_delta["delta_KGE_mapping"]=mapping_delta["official_station_alternative"]-mapping_delta["robin_original"]
    mapping_delta.to_csv(TABLE_DIR/f"{label}_original_vs_alternative_mapping_delta_KGE.csv",index=False)
    metrics.to_csv(TABLE_DIR/f"{label}_station_metrics_all_available.csv",index=False); supplementary_summary.to_csv(TABLE_DIR/f"{label}_supplementary_all_available_summary.csv",index=False); summary.to_csv(TABLE_DIR/f"{label}_primary_9_paired_summary.csv",index=False)
    return metrics,summary,missing

def calibration_ranking(summary: pd.DataFrame, baseline_id: str) -> pd.DataFrame:
    cal=summary[(summary["period"].eq("calibration"))&(summary["mapping_mode"].eq(SELECTION_MAPPING_MODE))]
    pivot=cal.pivot(index="scenario_id",columns="dataset",values="mean_KGE").reset_index().rename(columns={"flow":"flow_mean_KGE","nitrate":"nitrate_mean_KGE"})
    base=pivot[pivot["scenario_id"].eq(baseline_id)]
    if base.empty: raise RuntimeError(f"Baseline {baseline_id} missing from calibration summary")
    bf=float(base.iloc[0]["flow_mean_KGE"]); bn=float(base.iloc[0]["nitrate_mean_KGE"])
    pivot["flow_delta"]=pivot["flow_mean_KGE"]-bf; pivot["nitrate_delta"]=pivot["nitrate_mean_KGE"]-bn
    pivot["joint_score"]=JOINT_FLOW_WEIGHT*pivot["flow_mean_KGE"]+JOINT_NITRATE_WEIGHT*pivot["nitrate_mean_KGE"]
    pivot["eligible"]=(pivot["flow_delta"]>=FLOW_MEAN_DELTA_FLOOR)&(pivot["nitrate_delta"]>=NITRATE_MEAN_DELTA_FLOOR)
    pivot["selection_rank_rule"]="eligible -> flow_mean_KGE -> nitrate_mean_KGE"
    return pivot.sort_values(["eligible","flow_mean_KGE","nitrate_mean_KGE"],ascending=[False,False,False])


## 8. Phase A selection and Phase B intensity-factor grid

**Dissertation correspondence:** Sections 2.5 and 2.6.

**What this step does:** Retains one tested direction per calibration domain, then combines their H, W and N strengths in the predefined 5 × 5 × 4 grid

**Checks / outputs:** Direction rankings, 100 Phase-B specifications and no-rollback/complete-vector audits


In [ ]:
# Framework correspondence: Sections 2.5 and 2.6.
# What this cell does (Phase A selection and Phase B construction): Retains one tested direction per domain and constructs the 100-combination H/W/N intensity grid.
# Reproducibility note: outputs and assertions in this cell are retained as executable evidence.

phase_a_metrics, phase_a_summary, phase_a_missing = evaluate_specs(
    phase_a_specs, "phase_A_complete_H_5y"
)
selected_module_directions = {}
direction_rows = []

if not phase_a_missing and not phase_a_summary.empty:
    phase_a_rank = calibration_ranking(phase_a_summary, "A00_BASELINE").merge(
        phase_a_grid[[
            "scenario_id", "direction_id", "module", "physical_hypothesis",
            "regionalised_anchor", "screening_level", "endpoint_strength",
            "parameters_json", "endpoint_parameters_json",
        ]],
        on="scenario_id", how="left", validate="one_to_one",
    )
    baseline_rank = phase_a_rank[phase_a_rank["scenario_id"].eq("A00_BASELINE")].iloc[0]
    transparent_rankings = []
    for module in ["hydrology", "wwtw", "soil_nitrate"]:
        candidates = phase_a_rank[phase_a_rank["module"].eq(module)].copy()
        if module == "hydrology":
            candidates["module_gate_pass"] = (
                candidates["nitrate_delta"] >= HYDRO_SCREEN_NITRATE_DELTA_FLOOR
            )
            target_metric = "flow_mean_KGE"
            criterion = "best tested H level by calibration flow KGE subject to the nitrate gate"
        else:
            candidates["module_gate_pass"] = (
                candidates["flow_delta"] >= WQ_SCREEN_FLOW_DELTA_FLOOR
            )
            target_metric = "nitrate_mean_KGE"
            criterion = "best tested W/N level by calibration nitrate KGE subject to the flow gate"

        candidates["target_improves_baseline"] = (
            candidates[target_metric] > float(baseline_rank[target_metric])
        )
        candidates["admissible_for_direction_selection"] = (
            candidates["module_gate_pass"] & candidates["target_improves_baseline"]
        )
        secondary_metric = "nitrate_mean_KGE" if module == "hydrology" else "flow_mean_KGE"
        candidates = candidates.sort_values(
            ["admissible_for_direction_selection", "module_gate_pass", target_metric,
             secondary_metric, "direction_id", "screening_level"],
            ascending=[False, False, False, False, True, True],
        ).reset_index(drop=True)
        candidates["selection_order"] = np.arange(1, len(candidates) + 1)
        admissible = candidates[candidates["admissible_for_direction_selection"]]

        if admissible.empty:
            chosen = phase_a_grid[phase_a_grid["scenario_id"].eq("A00_BASELINE")].iloc[0]
            selection_reason = "baseline retained: no tested strength both improved its target and passed the module gate"
        else:
            chosen = admissible.iloc[0]
            selection_reason = criterion

        selected_screening_scenario = str(chosen["scenario_id"])
        candidates["selected"] = candidates["scenario_id"].eq(selected_screening_scenario)
        candidates["selection_or_exclusion_reason"] = np.select(
            [
                candidates["selected"],
                ~candidates["module_gate_pass"],
                ~candidates["target_improves_baseline"],
            ],
            [
                selection_reason,
                "excluded: module protection gate failed",
                "excluded: target metric did not exceed baseline",
            ],
            default="admissible but outranked by the selected tested level",
        )
        transparent_rankings.append(candidates)

        selected_module_directions[module] = {
            "direction_id": str(chosen["direction_id"]),
            "selected_screening_scenario": selected_screening_scenario,
            "selected_screening_level": float(chosen["screening_level"]),
            "regionalised_anchor": str(chosen["regionalised_anchor"]),
            "endpoint_strength": float(chosen["endpoint_strength"]),
            "parameters": {
                key: float(value)
                for key, value in json.loads(chosen["endpoint_parameters_json"]).items()
            },
        }
        direction_rows.append({
            "module": module,
            "selected_direction": str(chosen["direction_id"]),
            "selected_screening_scenario": selected_screening_scenario,
            "selected_screening_level": float(chosen["screening_level"]),
            "regionalised_anchor": str(chosen["regionalised_anchor"]),
            "endpoint_strength": float(chosen["endpoint_strength"]),
            "selection_reason": selection_reason,
            "target_metric": target_metric,
        })

    direction_table = pd.DataFrame(direction_rows)
    phase_a_transparent_ranking = pd.concat(transparent_rankings, ignore_index=True)
    direction_table.to_csv(TABLE_DIR / "03_phase_A_selected_complete_directions.csv", index=False)
    phase_a_transparent_ranking.to_csv(
        TABLE_DIR / "03_phase_A_complete_direction_ranking.csv", index=False
    )
    display(direction_table)
    display(phase_a_transparent_ranking[[
        "module", "selection_order", "scenario_id", "direction_id", "screening_level",
        "flow_mean_KGE", "nitrate_mean_KGE", "flow_delta", "nitrate_delta",
        "module_gate_pass", "target_improves_baseline",
        "admissible_for_direction_selection", "selected", "selection_or_exclusion_reason",
    ]])
else:
    print("Phase-A outputs incomplete; Phase B cannot be built. Missing:", phase_a_missing)


def weighted_module_parameters(module, weight):
    """Return one explicit module state without separating regional and continuous H."""
    selected = selected_module_directions[module]
    target = selected["parameters"]
    weighted = dict(BASE_PARAMETERS)

    if module == "hydrology":
        endpoint = float(selected["endpoint_strength"])
        if endpoint <= 0:
            return weighted
        fraction = float(weight) / endpoint
    else:
        fraction = float(weight)

    for parameter, rule in PARAMETER_RULES.items():
        if rule["group"] != module or parameter == "regionalised_rule_index":
            continue
        value = BASE_PARAMETERS[parameter] + fraction * (
            target[parameter] - BASE_PARAMETERS[parameter]
        )
        if not (float(rule["lower"]) <= value <= float(rule["upper"])):
            return None
        weighted[parameter] = float(value)

    if module == "hydrology":
        anchor = selected["regionalised_anchor"]
        if anchor != "R00" and float(weight) > 0:
            key = (anchor, float(weight))
            if key not in REGIONALISED_WEIGHT_INDEX:
                return None
            weighted["regionalised_rule_index"] = float(REGIONALISED_WEIGHT_INDEX[key])
        else:
            weighted["regionalised_rule_index"] = 0.0
    return weighted


# Fail before Phase B materialisation if the coupled endpoints are not exact.
module_scaling_rows = []
for module, levels in {
    "hydrology": PHASE_B_H_WEIGHTS,
    "wwtw": PHASE_B_W_WEIGHTS,
    "soil_nitrate": PHASE_B_N_WEIGHTS,
}.items():
    for level in levels:
        vector = weighted_module_parameters(module, level)
        if vector is None:
            module_scaling_rows.append({
                "module": module, "level": level, "available": False,
                "regionalised_rule": None, "endpoint_exact": False,
            })
            continue
        regional_index = int(round(vector["regionalised_rule_index"]))
        endpoint_exact = True
        if module == "hydrology" and np.isclose(level, HYDROLOGY_ENDPOINT_STRENGTH):
            target = selected_module_directions[module]["parameters"]
            endpoint_exact = all(
                np.isclose(vector[name], target[name], rtol=1e-12, atol=1e-12)
                for name, rule in PARAMETER_RULES.items()
                if rule["group"] == "hydrology"
            )
        expected_wwtw_constant = np.nan
        if module == "wwtw":
            target = selected_module_directions[module]["parameters"]
            expected_wwtw_constant = (
                BASE_PARAMETERS["wwtw_nitrate_constant"]
                + float(level) * (
                    target["wwtw_nitrate_constant"]
                    - BASE_PARAMETERS["wwtw_nitrate_constant"]
                )
            )
            endpoint_exact = np.isclose(
                vector["wwtw_nitrate_constant"], expected_wwtw_constant,
                rtol=1e-12, atol=1e-12,
            )
        module_scaling_rows.append({
            "module": module,
            "level": level,
            "available": True,
            "regionalised_rule": REGIONALISED_CANDIDATE_BY_INDEX[regional_index],
            "endpoint_exact": bool(endpoint_exact),
            "expected_wwtw_nitrate_constant": expected_wwtw_constant,
            "surface_coefficient_multiplier": vector["surface_coefficient_multiplier"],
            "ihacres_p_multiplier": vector["ihacres_p_multiplier"],
            "land_residence_multiplier": vector["land_residence_multiplier"],
            "percolation_coefficient_multiplier": vector["percolation_coefficient_multiplier"],
            "soil_storage_multiplier": vector["soil_storage_multiplier"],
            "groundwater_residence_multiplier": vector["groundwater_residence_multiplier"],
            "groundwater_storage_multiplier": vector["groundwater_storage_multiplier"],
            "wwtw_nitrate_constant": vector["wwtw_nitrate_constant"],
            "soil_N_process_multiplier": vector["minfpar_N_process_multiplier"],
            "soil_N_denpar": vector["denpar"],
        })
module_scaling_audit = pd.DataFrame(module_scaling_rows)
if not module_scaling_audit["available"].all():
    raise RuntimeError("At least one declared Phase-B complete module level is unavailable")
if not module_scaling_audit["endpoint_exact"].all():
    raise RuntimeError("Complete-H endpoint or selected-direction W interpolation assertion failed")
module_scaling_audit.to_csv(TABLE_DIR / "04_phase_B_complete_module_scaling_audit.csv", index=False)
display(module_scaling_audit)

# N0 is not the corrected parameter baseline: it is the mandatory pre-processing
# regionalised field with neutral Phase-A process strength.
n0_vector = weighted_module_parameters("soil_nitrate", 0.0)
if n0_vector is None:
    raise RuntimeError("Soil-N N0 vector is unavailable")
_, n0_full_parameters, n0_runtime, _ = build_scenario_config("CONTRACT_N0_PREPROCESSING", n0_vector)
n0_rows = []
for node_name, preprocessing_factor in SOIL_N_PREPROCESSING_FACTOR_BY_NODE.items():
    expected = SOIL_N_REFERENCE_MINFPAR * preprocessing_factor
    values = n0_runtime["minfpar_N_by_node_group"][node_name]
    passed = all(np.isclose(values[group], expected, rtol=1e-12, atol=1e-12) for group in SURFACE_GROUPS)
    n0_rows.append({
        "node": node_name,
        "preprocessing_regionalisation_factor": preprocessing_factor,
        "expected_N0_minfpar_N": expected,
        "actual_N0_minfpar_N": values[next(iter(SURFACE_GROUPS))],
        "process_multiplier": n0_full_parameters["minfpar_N_process_multiplier"],
        "denpar": n0_full_parameters["denpar"],
        "N0_equals_preprocessing_not_corrected_baseline": passed,
    })
n0_inheritance_audit = pd.DataFrame(n0_rows)
if (
    not n0_inheritance_audit["N0_equals_preprocessing_not_corrected_baseline"].all()
    or not np.isclose(n0_full_parameters["minfpar_N_process_multiplier"], 1.0)
    or not np.isclose(n0_full_parameters["denpar"], 0.015)
):
    raise RuntimeError("Phase-B Soil-N N0 rollback-prevention contract failed")
n0_inheritance_audit.to_csv(TABLE_DIR / "04_phase_B_soil_N_N0_no_rollback_audit.csv", index=False)
display(n0_inheritance_audit)


phase_b_design = phase_b_specs = pd.DataFrame()
phase_b_excluded_rows = []
if not phase_a_missing and not phase_a_summary.empty:
    module_names = ["hydrology", "wwtw", "soil_nitrate"]
    phase_b_rows = []
    for h_weight, w_weight, n_weight in itertools.product(
        PHASE_B_H_WEIGHTS, PHASE_B_W_WEIGHTS, PHASE_B_N_WEIGHTS
    ):
        weights = {
            "hydrology": float(h_weight),
            "wwtw": float(w_weight),
            "soil_nitrate": float(n_weight),
        }
        module_vectors = {
            module: weighted_module_parameters(module, weight)
            for module, weight in weights.items()
        }
        if any(vector is None for vector in module_vectors.values()):
            phase_b_excluded_rows.append({
                **weights,
                "reason": "complete module vector unavailable or outside physical bounds",
            })
            continue
        params = dict(BASE_PARAMETERS)
        for module in module_names:
            for parameter, rule in PARAMETER_RULES.items():
                if rule["group"] == module:
                    params[parameter] = module_vectors[module][parameter]

        if all(np.isclose(weights[module], 0.0) for module in module_names):
            scenario_id = "B00_BASELINE"
        else:
            scenario_id = (
                f"B_H{int(round(h_weight * 100)):03d}"
                f"_W{int(round(w_weight * 100)):03d}"
                f"_N{int(round(n_weight * 100)):03d}"
            )
        phase_b_rows.append({
            "scenario_id": scenario_id,
            "module": "complete_vector_combination",
            "hydrology_weight": h_weight,
            "wwtw_weight": w_weight,
            "soil_nitrate_weight": n_weight,
            "hydrology_direction": selected_module_directions["hydrology"]["direction_id"],
            "wwtw_direction": selected_module_directions["wwtw"]["direction_id"],
            "soil_nitrate_direction": selected_module_directions["soil_nitrate"]["direction_id"],
            "parameters_json": json.dumps(params, sort_keys=True),
        })

    phase_b_grid_all = pd.DataFrame(phase_b_rows)
    phase_b_duplicates = phase_b_grid_all[
        phase_b_grid_all.duplicated("parameters_json", keep="first")
    ].copy()
    phase_b_duplicates.to_csv(TABLE_DIR / "04_phase_B_skipped_duplicate_parameter_sets.csv", index=False)
    pd.DataFrame(phase_b_excluded_rows).to_csv(
        TABLE_DIR / "04_phase_B_excluded_complete_vector_combinations.csv", index=False
    )
    phase_b_grid = phase_b_grid_all.reset_index(drop=True)
    phase_b_grid.to_csv(TABLE_DIR / "04_phase_B_complete_vector_grid.csv", index=False)
    phase_b_design, phase_b_changes, phase_b_audits, phase_b_specs = materialise_scenarios(
        phase_b_grid, "phase_B_complete_H_5y", COARSE_WARMUP_START, COARSE_WARMUP_END
    )
    print("Phase-B complete-vector combinations:", len(phase_b_design))
    display(phase_b_grid[[
        "scenario_id", "hydrology_weight", "wwtw_weight", "soil_nitrate_weight",
        "hydrology_direction",
    ]])

if RUN_PHASE_B:
    if phase_b_specs.empty:
        raise RuntimeError("Complete Phase A before Phase B")
    phase_b_run_results = run_specs(phase_b_specs, "phase_B_complete_H_5y")
    display(phase_b_run_results)
else:
    print("Phase B not launched.")


## 9. Phase B selection and catchment-level assessment

**Dissertation correspondence:** Sections 2.6 and 2.7.

**What this step does:** Selects the best-tested combined configuration only from calibration-period mean KGE, then reports its spatial response across monitored catchments

**Checks / outputs:** Phase-B ranking, selected-versus-baseline catchment KGE changes, and final multi-metric tables


In [ ]:
# Framework correspondence: Sections 2.6 and 2.7.
# What this cell does: select one Phase-B combination using calibration-period
# mean KGE only, then export descriptive catchment-level changes.  It does not
# alter the selected configuration after Phase B.
phase_b_metrics, phase_b_summary, phase_b_missing = (pd.DataFrame(), pd.DataFrame(), [])
if not phase_b_specs.empty:
    phase_b_metrics, phase_b_summary, phase_b_missing = evaluate_specs(
        phase_b_specs, "phase_B_complete_H_5y"
    )

phase_b_rank = pd.DataFrame()
global_phase_b_scenario = None
global_phase_b_row = None
final_selected_scenario = None
final_status = "not_evaluated"

if not phase_b_missing and not phase_b_summary.empty:
    phase_b_rank = calibration_ranking(phase_b_summary, "B00_BASELINE").merge(
        phase_b_design[[
            "scenario_id", "parameters_json", "hydrology_weight",
            "wwtw_weight", "soil_nitrate_weight", "config_path",
        ]],
        on="scenario_id", how="left", validate="one_to_one",
    )
    phase_b_rank = phase_b_rank.reset_index(drop=True)
    phase_b_rank["selection_order"] = np.arange(1, len(phase_b_rank) + 1)
    eligible = phase_b_rank[phase_b_rank["eligible"]]
    global_phase_b_row = (
        eligible.iloc[0]
        if not eligible.empty
        else phase_b_rank[phase_b_rank["scenario_id"].eq("B00_BASELINE")].iloc[0]
    )
    global_phase_b_scenario = str(global_phase_b_row["scenario_id"])
    phase_b_rank["selected_global"] = phase_b_rank["scenario_id"].eq(global_phase_b_scenario)
    phase_b_rank["selection_or_exclusion_reason"] = np.select(
        [
            phase_b_rank["selected_global"] & phase_b_rank["eligible"],
            phase_b_rank["selected_global"] & ~phase_b_rank["eligible"],
            ~phase_b_rank["eligible"],
        ],
        [
            "selected: first eligible row under flow-first lexicographic ranking",
            "baseline retained: no candidate improved both calibration-period mean KGE values",
            "excluded: failed the dual mean-KGE improvement criterion",
        ],
        default="eligible but outranked by flow mean KGE, then nitrate mean KGE",
    )
    phase_b_rank.to_csv(TABLE_DIR / "05_phase_B_complete_vector_ranking.csv", index=False)
    print("Selected Phase-B configuration:", global_phase_b_scenario)
    display(phase_b_rank)

if global_phase_b_scenario is None:
    raise RuntimeError(f"Complete Phase B before selection. Missing: {phase_b_missing}")

# Section 2.7: spatial consistency is an assessment, not an additional
# calibration stage.  Positive/negative changes are reported per metric.
primary_cal = phase_b_metrics[
    phase_b_metrics["mapping_mode"].eq(SELECTION_MAPPING_MODE)
    & phase_b_metrics["period"].eq("calibration")
    & phase_b_metrics["catchment_id"].astype(str).isin(PRIMARY_PAIRED_WFDS)
].copy()
baseline_cal = primary_cal[primary_cal["scenario_id"].eq("B00_BASELINE")][
    ["catchment_id", "dataset", "KGE"]
].rename(columns={"KGE": "baseline_KGE"})
selected_cal = primary_cal[primary_cal["scenario_id"].eq(global_phase_b_scenario)][
    ["catchment_id", "dataset", "KGE"]
].rename(columns={"KGE": "selected_KGE"})
phase_b_selected_wfd_delta = baseline_cal.merge(
    selected_cal, on=["catchment_id", "dataset"], validate="one_to_one"
)
phase_b_selected_wfd_delta["delta_KGE"] = (
    phase_b_selected_wfd_delta["selected_KGE"]
    - phase_b_selected_wfd_delta["baseline_KGE"]
)
phase_b_selected_wfd_delta["response_direction"] = np.where(
    phase_b_selected_wfd_delta["delta_KGE"] > MIN_WFD_POSITIVE_DELTA,
    "improved",
    np.where(
        phase_b_selected_wfd_delta["delta_KGE"] < -MIN_WFD_POSITIVE_DELTA,
        "deteriorated",
        "no material change",
    ),
)
phase_b_selected_wfd_delta.to_csv(
    TABLE_DIR / "06_phase_B_selected_WFD_delta.csv", index=False
)
display(phase_b_selected_wfd_delta)

final_selected_scenario = global_phase_b_scenario
final_status = (
    "selected from the predefined Phase-B intensity-factor grid using "
    "calibration-period mean flow KGE followed by mean nitrate KGE"
)
print("Final framework selection:", final_selected_scenario, "—", final_status)


## 10. Final reporting and untouched validation

**Dissertation correspondence:** Section 2.7.

**What this step does:** Holds the Phase-B selection fixed and compares it with B00 in both calibration and independent validation periods

**Checks / outputs:** Final selection contract, summary table and catchment-level diagnostic outputs


In [ ]:
# Framework correspondence: Section 2.7.
# What this cell does: reports the selected Phase-B configuration at every
# monitored catchment and evaluates the already-fixed configuration during the
# untouched validation period.  No validation value is used for selection.
assert_sources_unchanged()

final_comparison = pd.DataFrame()
final_wfd_detail = pd.DataFrame()

if final_selected_scenario is not None:
    baseline_output = RUN_DIR / "B00_BASELINE" / "model_outputs_2014_2019.csv"
    selected_output = RUN_DIR / final_selected_scenario / "model_outputs_2014_2019.csv"
    baseline_all = evaluate_scenario("B00_BASELINE", baseline_output)
    selected_all = evaluate_scenario(final_selected_scenario, selected_output)
    baseline = baseline_all[
        baseline_all["catchment_id"].astype(str).isin(PRIMARY_PAIRED_WFDS)
        & baseline_all["mapping_mode"].eq(SELECTION_MAPPING_MODE)
    ][["period", "dataset", "catchment_id", "KGE", "r", "alpha", "beta"]].copy()
    selected = selected_all[
        selected_all["catchment_id"].astype(str).isin(PRIMARY_PAIRED_WFDS)
        & selected_all["mapping_mode"].eq(SELECTION_MAPPING_MODE)
    ][["period", "dataset", "catchment_id", "KGE", "r", "alpha", "beta"]].copy()
    final_wfd_detail = baseline.merge(
        selected,
        on=["period", "dataset", "catchment_id"],
        suffixes=("_baseline", "_selected"),
        validate="one_to_one",
    )
    final_wfd_detail["delta_KGE"] = (
        final_wfd_detail["KGE_selected"] - final_wfd_detail["KGE_baseline"]
    )
    final_wfd_detail["response_direction"] = np.where(
        final_wfd_detail["delta_KGE"] > MIN_WFD_POSITIVE_DELTA,
        "improved",
        np.where(
            final_wfd_detail["delta_KGE"] < -MIN_WFD_POSITIVE_DELTA,
            "deteriorated",
            "no material change",
        ),
    )
    final_wfd_detail.to_csv(TABLE_DIR / "FINAL_WFD_KGE_DETAIL.csv", index=False)

    final_comparison = final_wfd_detail.groupby(
        ["period", "dataset"], as_index=False
    ).agg(
        baseline_mean_KGE=("KGE_baseline", "mean"),
        selected_mean_KGE=("KGE_selected", "mean"),
        mean_delta_KGE=("delta_KGE", "mean"),
        worst_delta_KGE=("delta_KGE", "min"),
        finite_WFDs=("KGE_selected", "count"),
        improved_WFDs=("response_direction", lambda values: int((values == "improved").sum())),
        deteriorated_WFDs=("response_direction", lambda values: int((values == "deteriorated").sum())),
    )
    final_comparison.insert(0, "selected_scenario", final_selected_scenario)
    final_comparison.to_csv(TABLE_DIR / "FINAL_BASELINE_SELECTED_SUMMARY.csv", index=False)
    display(final_comparison)
    display(final_wfd_detail)
else:
    raise RuntimeError("Phase-B selection was not completed; final reporting is unavailable.")


run_contract = {
    "framework_implementation_version": FRAMEWORK_IMPLEMENTATION_VERSION,
    "workflow_root": str(WORKFLOW_ROOT),
    "architecture": "corrected national baseline -> evidence-informed pre-processing regionalisation -> separate-domain Phase A -> combined Phase B intensity grid -> multi-site assessment and untouched validation",
    "time_contract": {
        "warmup": [COARSE_WARMUP_START, COARSE_WARMUP_END],
        "calibration": [CALIBRATION_START, CALIBRATION_END],
        "validation": [VALIDATION_START, VALIDATION_END],
        "validation_used_for_selection": False,
    },
    "complete_H_definition": {
        "endpoint_strength": HYDROLOGY_ENDPOINT_STRENGTH,
        "regionalised_anchors": [f"R{i:02d}" for i in range(1, 8)],
        "process_bundles": HYDROLOGY_PROCESS_BUNDLES,
        "phase_A_screening_levels": PHASE_A_H_SCREEN_LEVELS,
        "phase_B_H_weights": PHASE_B_H_WEIGHTS,
        "normalisation": "continuous H delta is multiplied by H/1.75; regional field uses the matching anchor strength H",
    },
    "phase_B_weights": {
        "H": PHASE_B_H_WEIGHTS,
        "W": PHASE_B_W_WEIGHTS,
        "N": PHASE_B_N_WEIGHTS,
    },
    "soil_N_preprocessing_definition": {
        "shared_function": "node9056_regionalisation_generator_v1_1_0.thresholded_multiplier",
        "mixture_threshold": SOIL_N_MIXTURE_THRESHOLD,
        "dominant_threshold": SOIL_N_DOMINANT_THRESHOLD,
        "class_multipliers_mi": SOIL_N_PRIOR_MULTIPLIERS,
        "whole_model_normalisation_applied": False,
        "N0_semantics": "mandatory pre-processing regionalised minfpar_N field with process multiplier 1 and denpar 0.015",
    },
    "phase_A_screening_levels": {
        "H": PHASE_A_H_SCREEN_LEVELS,
        "W": PHASE_A_W_SCREEN_LEVELS,
        "N": PHASE_A_N_SCREEN_LEVELS,
    },
    "phase_B_selection_rule": "candidates improving both calibration-period mean KGE values -> flow_mean_KGE -> nitrate_mean_KGE; joint_score report-only",
    "selected_phase_A_directions": selected_module_directions,
    "selected_phase_B_scenario": final_selected_scenario,
    "final_status": final_status,
    "selection_mapping": SELECTION_MAPPING_MODE,
    "primary_WFDs": PRIMARY_PAIRED_WFDS,
    "source_contract_sha256": SOURCE_CONTRACT_SHA256,
}
(WORKFLOW_ROOT / "RUN_CONTRACT_AND_FINAL_SELECTION.json").write_text(
    json.dumps(run_contract, indent=2), encoding="utf-8"
)

print(json.dumps(run_contract, indent=2))
print("\nKey outputs:")
for path in [
    TABLE_DIR / "02_phase_A_coupled_direction_library.csv",
    TABLE_DIR / "02_phase_A_complete_H_contract_audit.csv",
    TABLE_DIR / "02_phase_A_soil_N_preprocessing_inheritance_audit.csv",
    TABLE_DIR / "03_phase_A_complete_direction_ranking.csv",
    TABLE_DIR / "03_phase_A_selected_complete_directions.csv",
    TABLE_DIR / "04_phase_B_complete_vector_grid.csv",
    TABLE_DIR / "04_phase_B_complete_module_scaling_audit.csv",
    TABLE_DIR / "04_phase_B_soil_N_N0_no_rollback_audit.csv",
    TABLE_DIR / "05_phase_B_complete_vector_ranking.csv",
    TABLE_DIR / "06_phase_B_selected_WFD_delta.csv",
    TABLE_DIR / "FINAL_BASELINE_SELECTED_SUMMARY.csv",
    TABLE_DIR / "FINAL_WFD_KGE_DETAIL.csv",
    WORKFLOW_ROOT / "RUN_CONTRACT_AND_FINAL_SELECTION.json",
]:
    print(" -", "exists" if path.exists() else "pending", path)
